**## 0. 공통 유틸 (2026.7.1 광주·전남 통합 반영)**

기존 KOSIS/SGIS 원자료(자동차등록대수 등)는 광주광역시/전라남도를 아직 따로따로 표기하는데, 가연 버스 데이터와 은규 지형장벽 데이터(2026-07-01 경계 기준)는 이미 통합된 "전남광주통합특별시"로 표기돼 있다. `resolve_sido()`를 모든 지역명 처리 지점에 일관되게 넣어두면, 어느 쪽이 옛날 표기든 새 표기든 자동으로 같은 키로 수렴한다(실제로 돌려서 확인함: "광주광역시 동구"/"전라남도 목포시"가 자동으로 "전남광주통합특별시 동구"/"전남광주통합특별시 목포시"로 변환됨).

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

SIDO_NAMES = {
    '서울특별시','부산광역시','대구광역시','인천광역시','광주광역시','대전광역시','울산광역시',
    '세종특별자치시','경기도','강원특별자치도','충청북도','충청남도','전북특별자치도',
    '전라남도','경상북도','경상남도','제주특별자치도','전남광주통합특별시',
}
SIDO_ALIASES = {
    '강원도': '강원특별자치도',
    '전라북도': '전북특별자치도',
    '광주광역시': '전남광주통합특별시',   # 2026.7.1 광주·전남 통합
    '전라남도': '전남광주통합특별시',     # 2026.7.1 광주·전남 통합
}
NON_REGION = {'전국', '동부', '읍부', '면부'}
GU_HAVING_CITIES = {'수원시','성남시','안양시','안산시','고양시','용인시',
                     '청주시','천안시','전주시','포항시','창원시','부천시'}
BARE_GU_TO_CITY = {
    ('경기도','장안구'):'수원시', ('경기도','권선구'):'수원시', ('경기도','팔달구'):'수원시', ('경기도','영통구'):'수원시',
    ('경기도','수정구'):'성남시', ('경기도','중원구'):'성남시', ('경기도','분당구'):'성남시',
    ('경기도','만안구'):'안양시', ('경기도','동안구'):'안양시',
    ('경기도','상록구'):'안산시', ('경기도','단원구'):'안산시',
    ('경기도','덕양구'):'고양시', ('경기도','일산동구'):'고양시', ('경기도','일산서구'):'고양시',
    ('경기도','처인구'):'용인시', ('경기도','기흥구'):'용인시', ('경기도','수지구'):'용인시',
    ('경기도','원미구'):'부천시', ('경기도','소사구'):'부천시', ('경기도','오정구'):'부천시',
    ('충청북도','상당구'):'청주시', ('충청북도','서원구'):'청주시', ('충청북도','흥덕구'):'청주시', ('충청북도','청원구'):'청주시',
    ('충청남도','동남구'):'천안시', ('충청남도','서북구'):'천안시',
    ('전북특별자치도','완산구'):'전주시', ('전북특별자치도','덕진구'):'전주시',
    ('경상북도','남구'):'포항시', ('경상북도','북구'):'포항시',
    ('경상남도','의창구'):'창원시', ('경상남도','성산구'):'창원시',
    ('경상남도','마산합포구'):'창원시', ('경상남도','마산회원구'):'창원시', ('경상남도','진해구'):'창원시',
}
LEGACY_SIDO_OVERRIDE = {
    ('경상북도','군위군'): '대구광역시',   # 2023년 대구 편입
}

def resolve_sido(name):
    """예전 명칭(광주광역시/전라남도/강원도/전라북도)을 현재 명칭으로 통일."""
    if pd.isna(name):
        return None
    return SIDO_ALIASES.get(str(name).strip(), str(name).strip())

def canonical_key(sido, sigungu):
    sido = resolve_sido(sido)
    if sido is None:
        return None
    if '세종' in sido:
        return '세종특별자치시'
    return f"{sido} {sigungu}"

def normalize_sgg_for_join(sido, sgg):
    """특례시 하위구 표기('수원시장안구' 또는 시 이름 없이 '북구'만 있는 경우 등)를 상위 시로 되돌린다."""
    if pd.isna(sgg) or str(sgg).strip() == '':
        return None
    sido_r = resolve_sido(sido)
    name = str(sgg).strip()
    if (sido_r, name) in BARE_GU_TO_CITY:
        return BARE_GU_TO_CITY[(sido_r, name)]
    name_nospace = name.replace(' ', '')
    for city in GU_HAVING_CITIES:
        if name_nospace.startswith(city):
            return city
    return name

def broadcast_sigungu_to_eupmyeon(eupmyeon_df, sigungu_score_df, sido_col='sidonm', sgg_col='sggnm', on='sigungu'):
    """시군구 단위 점수를 읍면동 단위 표에 그대로 복제해서 붙인다 (자가용미보유용)."""
    df = eupmyeon_df.copy()
    def make_key(r):
        sido = resolve_sido(r[sido_col])
        sgg_norm = normalize_sgg_for_join(sido, r.get(sgg_col))
        override = LEGACY_SIDO_OVERRIDE.get((sido, sgg_norm))
        if override:
            sido = override
        return canonical_key(sido, sgg_norm if sgg_norm else sido)
    df['_join_key'] = df.apply(make_key, axis=1)
    merged = df.merge(sigungu_score_df, left_on='_join_key', right_on=on, how='left')
    return merged.drop(columns=['_join_key'])

def build_match_key(sido, sgg, dong):
    """읍면동 고유 데이터끼리(의료·버스 등) 조인할 때 쓰는 공통 키."""
    sido_r = resolve_sido(sido)
    sgg_norm = normalize_sgg_for_join(sido_r, sgg) or sido_r
    return f"{sido_r}|{sgg_norm}|{str(dong).strip()}"

def minmax(s):
    """값이 클수록 나쁨(취약)인 지표를 그대로 0~1로."""
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn)

def minmax_invert(s):
    """값이 클수록 좋음(안 취약)인 지표를 뒤집어서 0~1 취약점수로."""
    mn, mx = s.min(), s.max()
    return 1 - (s - mn) / (mx - mn)

**## 1. 자가용미보유 (시군구 단위 계산)**

In [2]:
print('1) 1인당 자동차등록대수 CSV 업로드')
uploaded = files.upload()
car_path = list(uploaded.keys())[0]

def load_car_percap_sigungu(path):
    df = pd.read_csv(path, encoding='cp949')
    df = df.iloc[1:].reset_index(drop=True)
    df.columns = ['행정구역별', '1인당등록대수']
    df['1인당등록대수'] = pd.to_numeric(df['1인당등록대수'], errors='coerce')

    keys, values = [], []
    cur_sido = None
    skip_sub_gu = False
    for raw, val in zip(df['행정구역별'], df['1인당등록대수']):
        name = str(raw).strip()
        if name == '전남광주통합특별시' and pd.isna(val):
            continue  # 값 없는 표기용 헤더 행 (실제 광주/전라남도는 아래에서 각각 처리됨 -> 자동으로 통합명으로 변환됨)
        if name in NON_REGION:
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            if '세종' in canon and pd.notna(val):
                keys.append('세종특별자치시'); values.append(val)
            cur_sido, skip_sub_gu = canon, False
            continue
        if skip_sub_gu and name.endswith('구'):
            continue
        skip_sub_gu = False
        if name.endswith(('시', '군', '구')):
            if name in GU_HAVING_CITIES:
                skip_sub_gu = True
            if cur_sido:
                keys.append(canonical_key(cur_sido, name)); values.append(val)
            continue

    out = pd.DataFrame({'sigungu': keys, '1인당_자동차등록대수': values})
    return out.groupby('sigungu', as_index=False)['1인당_자동차등록대수'].mean()

car_sigungu = load_car_percap_sigungu(car_path)
car_sigungu['자가용미보유_취약점수'] = minmax_invert(car_sigungu['1인당_자동차등록대수'])
print(car_sigungu.shape, '(광주+전남 통합 후 228개 전후로 나와야 정상)')

1) 1인당 자동차등록대수 CSV 업로드


Saving 1인당_자동차_등록대수_시도_시_군_구__20260923121149.csv to 1인당_자동차_등록대수_시도_시_군_구__20260923121149.csv
(229, 3) (광주+전남 통합 후 228개 전후로 나와야 정상)


**## 2. 읍면동 마스터 테이블 (지형장벽 기준) + 자가용미보유 브로드캐스트**

In [3]:
print('2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)')
uploaded = files.upload()
eupmyeon_master = pd.read_csv(list(uploaded.keys())[0])
print('지형장벽 행 수:', len(eupmyeon_master))

eupmyeon_master = broadcast_sigungu_to_eupmyeon(eupmyeon_master, car_sigungu)
print('자가용미보유 결측:', eupmyeon_master['자가용미보유_취약점수'].isna().sum())
if eupmyeon_master['자가용미보유_취약점수'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['자가용미보유_취약점수'].isna()][['sidonm','sggnm','emdnm']].drop_duplicates())

2) slope_score.csv 업로드 (은규 지형장벽, 기준 테이블)


Saving slope_score.csv to slope_score.csv
지형장벽 행 수: 3555
자가용미보유 결측: 77
     sidonm   sggnm emdnm
914   인천광역시     서해구  가정2동
1181  인천광역시     서해구  석남3동
1220    경기도  화성시동탄구  동탄9동
1310    경기도  화성시동탄구  동탄5동
1393  인천광역시     영종구   용유동
...     ...     ...   ...
3471  인천광역시     서해구  청라1동
3473  인천광역시     영종구  운서1동
3482  인천광역시     영종구  영종1동
3498  인천광역시     서해구  청라2동
3542    경기도  화성시만세구   새솔동

[77 rows x 3 columns]


**## 2-보완. 고령인구비율 (KOSIS 101_DT_1IN1503, 읍면동 단위 재다운로드본)**
KOSIS에서 "행정구역별(읍면동)" 트리를 시군구 레벨까지만 펼친 채로 받으면 시군구 단위로만 나온다 — 트리를 읍면동까지 전부 펼치고, 항목을 "총인구(명)" 하나로 좁혀서(그래야 20만 셀 제한을 통과함) 다시 받은 파일이면 진짜 읍면동 단위다. 단, 이 KOSIS 원자료는 시도→시군구→읍면동 각 레벨마다 그 레벨 자체의 합계 행(예: "수원시" 전체 합계가 "장안구" 등 하위 구 목록이 나오기 전에 먼저 한 번 나옴)을 끼워 넣기 때문에, 단순히 "시/군/구로 안 끝나면 읍면동"으로 판단하면 특례시(수원시·성남시·안양시·청주시·천안시·전주시·포항시·창원시·부천시 등)의 시 전체 합계 행을 가짜 읍면동으로 잘못 잡아먹는다. 아래 코드는 연속된 동일 라벨을 하나의 "블록"으로 묶어서 이 문제를 피한다.

In [4]:
print('2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)')
uploaded = files.upload()
age_path = list(uploaded.keys())[0]

import xml.etree.ElementTree as ET

def read_kosis_xml_xls(path):
    """KOSIS가 .xls 확장자로 내려주지만 실제로는 XML(엑셀 2003 XML) 형식인 파일을 파싱."""
    with open(path, 'rb') as f:
        raw = f.read()
    text = raw.decode('euc-kr', errors='replace').lstrip()
    text = text.replace('encoding="EUC-KR"', 'encoding="UTF-8"')
    idx = text.find('메타정보')
    if idx != -1:
        idx2 = text.rfind('<Worksheet', 0, idx)
        text = text[:idx2] + '</Workbook>'
    root = ET.fromstring(text)
    ns = '{urn:schemas-microsoft-com:office:spreadsheet}'
    rows_out = []
    for ws in root.findall(f'{ns}Worksheet'):
        table = ws.find(f'{ns}Table')
        if table is None:
            continue
        for row in table.findall(f'{ns}Row'):
            cells = []
            for cell in row.findall(f'{ns}Cell'):
                data = cell.find(f'{ns}Data')
                cells.append(data.text if data is not None else None)
            rows_out.append(cells)
    return rows_out

def load_age_eupmyeondong(path):
    raw_rows = read_kosis_xml_xls(path)
    data_rows = [r for r in raw_rows[2:] if r and len(r) >= 5]
    df = pd.DataFrame(data_rows, columns=['행정구역_raw', '연령별', '항목', '단위', '2025'])
    df['2025'] = pd.to_numeric(df['2025'], errors='coerce')

    names = df['행정구역_raw'].astype(str).str.strip()
    block_id = (names != names.shift()).cumsum()        # 연속 동일 라벨을 하나의 블록으로
    block_labels = names.groupby(block_id).first()

    cur_sido, cur_sgg, cur_sgg_is_gu_city = None, None, False
    block_sido, block_sgg, block_dong = {}, {}, {}

    for bid, name in block_labels.items():
        if name in NON_REGION:
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        canon = resolve_sido(name)
        if canon in SIDO_NAMES:
            cur_sido, cur_sgg, cur_sgg_is_gu_city = canon, None, False
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith('구') and cur_sgg_is_gu_city:
            # 특례시 하위 자치구(예: 수원시 장안구) -> 구조 표시일 뿐, 실제 단위 아님
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        if name.endswith(('시', '군', '구')):
            cur_sgg = name
            cur_sgg_is_gu_city = name in GU_HAVING_CITIES
            block_sido[bid] = block_sgg[bid] = block_dong[bid] = None
            continue
        # 읍면동 레벨
        block_sido[bid] = cur_sido
        block_sgg[bid] = cur_sgg
        block_dong[bid] = name

    df['_block'] = block_id
    df['sido'] = df['_block'].map(block_sido)
    df['sgg'] = df['_block'].map(block_sgg)
    df['dong'] = df['_block'].map(block_dong)

    valid = df[df['dong'].notna()].copy()
    total = valid[valid['연령별'] == '합계'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='전체인구')
    elderly = valid[valid['연령별'] == '65세이상'].groupby(['sido', 'sgg', 'dong'])['2025'].sum().reset_index(name='고령인구')
    out = total.merge(elderly, on=['sido', 'sgg', 'dong'], how='left')
    out['고령인구비율'] = out['고령인구'] / out['전체인구']
    return out

age_eupmyeon = load_age_eupmyeondong(age_path)
print(age_eupmyeon.shape, '(3,555~3,558개 근처여야 정상)')
print('전체인구 합계 (전국 51,817,499명과 비교):', age_eupmyeon['전체인구'].sum())

age_eupmyeon['_match_key'] = age_eupmyeon.apply(lambda r: build_match_key(r['sido'], r['sgg'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    age_eupmyeon[['_match_key', '고령인구비율']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])
eupmyeon_master['고령인구비율_취약점수'] = minmax(eupmyeon_master['고령인구비율'])  # 비율이 높을수록 취약

print('고령인구비율 결측:', eupmyeon_master['고령인구비율'].isna().sum())
if eupmyeon_master['고령인구비율'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['고령인구비율'].isna()][['sidonm', 'sggnm', 'emdnm']].drop_duplicates().head(20))

2-보완) 고령인구 KOSIS 파일 업로드 (101_DT_1IN1503, 읍면동 단위 재다운로드본)


Saving 101_DT_1IN1503_20261004030052.xls to 101_DT_1IN1503_20261004030052.xls
(3557, 6) (3,555~3,558개 근처여야 정상)
전체인구 합계 (전국 51,817,499명과 비교): 51817499.0
고령인구비율 결측: 85
     sidonm   sggnm  emdnm
128     경기도  안양시만안구   병목안동
847   부산광역시    부산진구   당감1동
886     경기도  안양시만안구    명학동
914   인천광역시     서해구   가정2동
1088    경기도  용인시처인구    양지읍
1181  인천광역시     서해구   석남3동
1220    경기도  화성시동탄구   동탄9동
1310    경기도  화성시동탄구   동탄5동
1393  인천광역시     영종구    용유동
1510  인천광역시     서해구   가정3동
1586    경기도  화성시동탄구   동탄8동
1636    경기도  화성시효행구    봉담읍
1655  인천광역시     검단구  불로대곡동
1670  인천광역시     검단구   아라2동
1671  인천광역시     서해구    연희동
1682    경기도  화성시효행구    비봉면
1713  인천광역시     검단구    검단동
1748  인천광역시     서해구   가좌2동
1749    경기도  화성시동탄구   동탄7동
1891    경기도  화성시효행구    매송면


**## 3. 의료접근성 (HIRA 버전, 동 포함 전국 3,558개 — 0 대체 없음)**

팀에서 이미 만든 `medical_access_score_hira.csv`(저장소 경로 `data/processed/medical/`)는 동 지역까지 전국 읍면동 전부에 실제 계산값이 들어 있다. 예전에 쓰던 `medical_access_score.csv`는 마을조사표 기반이라 읍/면 1,404개뿐이어서 동 지역을 0(가장 양호)으로 채워야 했는데, 이 0 대체가 의료_취약점수의 엔트로피 가중치를 58.9%까지 끌어올린 원인으로 의심됐다. HIRA 버전을 쓰면 0 대체가 필요 없다.

HIRA 파일의 실제 컬럼은 `읍면_key, sidonm, sggnm, emdnm, 1차의료_최근접거리_km, 2차의료_최근접거리_km, 1차의료_취약점수_HIRA, 2차의료_취약점수_HIRA, 의료시설_취약점수_HIRA`이다. 최종 의료 점수는 이미 계산돼 있는 `의료시설_취약점수_HIRA`이고, 확인해보니 0.4 × 1차 + 0.6 × 2차와 완전히 일치한다(최대 차이 1e-16). 따라서 가중치를 코드에서 다시 정할 필요 없이 이 컬럼을 그대로 쓰면 된다. 이 파일은 `sidonm`/`sggnm`/`emdnm`이 은규 지형장벽 파일과 같은 2026.7.1 기준 표기(전남광주통합특별시, 서해구, 동탄구 등)라서 `읍면_key`로 바로 붙는다.

In [5]:
print('3) medical_access_score_hira.csv 업로드 (HIRA 기반 의료접근성, 동 포함 전국 3,558개)')
uploaded = files.upload()
medical_df = pd.read_csv(list(uploaded.keys())[0], encoding='utf-8-sig')
print('행 수:', len(medical_df), '/ 읍면_key 중복:', medical_df['읍면_key'].duplicated().sum())

# 0.4/0.6이 실제로 적용돼 있는지 확인 (0에 가까워야 정상)
check = (medical_df['의료시설_취약점수_HIRA']
         - (0.4 * medical_df['1차의료_취약점수_HIRA'] + 0.6 * medical_df['2차의료_취약점수_HIRA'])).abs().max()
print('0.4/0.6 재계산과의 최대 차이:', check)

medical_df = medical_df.rename(columns={'의료시설_취약점수_HIRA': '의료_취약점수'})
eupmyeon_master = eupmyeon_master.merge(
    medical_df[['읍면_key', '의료_취약점수']], on='읍면_key', how='left'
)
# 더 이상 fillna(0)을 하지 않는다. 결측이 남으면 그대로 확인한다.
print('의료_취약점수 결측:', eupmyeon_master['의료_취약점수'].isna().sum())
if eupmyeon_master['의료_취약점수'].isna().sum():
    print(eupmyeon_master[eupmyeon_master['의료_취약점수'].isna()][['sidonm', 'sggnm', 'emdnm']].drop_duplicates().head(20))

3) medical_access_score_hira.csv 업로드 (HIRA 기반 의료접근성, 동 포함 전국 3,558개)


Saving medical_access_score_hira.csv to medical_access_score_hira.csv
행 수: 3558 / 읍면_key 중복: 0
0.4/0.6 재계산과의 최대 차이: 1.1102230246251565e-16
의료_취약점수 결측: 0


**## 4. 버스 (정류장 인구대비 + 배차간격, 가연 2차 수정본)**
가연이 시도명 정규화(광주·전남 통합 반영)를 적용해서 올려준 최신 파일. `province`/`sigungu`/`dong`/`dong_code`/`population_status` 컬럼이 있고 `per10k`(정류장, 인구대비), `headway_min`(배차간격)을 바로 쓸 수 있다.

In [7]:
print('4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)')
uploaded = files.upload()
stop_df = pd.read_csv(list(uploaded.keys())[0])

print('4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)')
uploaded = files.upload()
headway_df = pd.read_csv(list(uploaded.keys())[0])

stop_df['버스정류장_취약점수'] = minmax_invert(stop_df['per10k'])          # 적을수록 취약
headway_df['배차간격_취약점수'] = minmax(headway_df['headway_min'])        # 길수록 취약

stop_df['_match_key'] = stop_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
headway_df['_match_key'] = headway_df.apply(lambda r: build_match_key(r['province'], r['sigungu'], r['dong']), axis=1)
eupmyeon_master['_match_key'] = eupmyeon_master.apply(lambda r: build_match_key(r['sidonm'], r['sggnm'], r['emdnm']), axis=1)

eupmyeon_master = eupmyeon_master.merge(
    stop_df[['_match_key', '버스정류장_취약점수']], on='_match_key', how='left'
).merge(
    headway_df[['_match_key', '배차간격_취약점수']], on='_match_key', how='left'
)
eupmyeon_master = eupmyeon_master.drop(columns=['_match_key'])

print('버스정류장_취약점수 결측 (대체 전):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 전):', eupmyeon_master['배차간격_취약점수'].isna().sum())

# 결측 여부를 먼저 플래그로 남겨둠 (보고서용)
eupmyeon_master['버스정류장_취약점수_대체여부'] = eupmyeon_master['버스정류장_취약점수'].isna()
eupmyeon_master['배차간격_취약점수_대체여부'] = eupmyeon_master['배차간격_취약점수'].isna()

def fill_by_region_mean(df, col, sgg_col='sggnm', sido_col='sidonm'):
    sgg_mean = df.groupby(sgg_col)[col].transform('mean')
    df[col] = df[col].fillna(sgg_mean)
    sido_mean = df.groupby(sido_col)[col].transform('mean')
    df[col] = df[col].fillna(sido_mean)
    return df

eupmyeon_master = fill_by_region_mean(eupmyeon_master, '버스정류장_취약점수')
eupmyeon_master = fill_by_region_mean(eupmyeon_master, '배차간격_취약점수')

print('버스정류장_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['버스정류장_취약점수'].isna().sum())
print('배차간격_취약점수 결측 (대체 후, 0이어야 정상):', eupmyeon_master['배차간격_취약점수'].isna().sum())
print('잠정 대체된 읍면동 수 — 버스정류장:', eupmyeon_master['버스정류장_취약점수_대체여부'].sum(),
      ' / 배차간격:', eupmyeon_master['배차간격_취약점수_대체여부'].sum())

4-1) stop_supply_by_dong.csv 업로드 (가연 버스정류장·인구대비, 2차 수정본)


Saving stop_supply_by_dong.csv to stop_supply_by_dong.csv
4-2) dong_supply.csv 업로드 (가연 배차간격, 2차 수정본)


Saving dong_supply.csv to dong_supply.csv
버스정류장_취약점수 결측 (대체 전): 144
배차간격_취약점수 결측 (대체 전): 13
버스정류장_취약점수 결측 (대체 후, 0이어야 정상): 0
배차간격_취약점수 결측 (대체 후, 0이어야 정상): 0
잠정 대체된 읍면동 수 — 버스정류장: 144  / 배차간격: 13


**## 5. 엔트로피 가중치 계산**

In [8]:
# 공급공백(완성되면) 나오는 대로 여기 추가
final_vars = ['지형장벽_취약점수', '자가용미보유_취약점수', '의료_취약점수',
              '버스정류장_취약점수', '배차간격_취약점수', '고령인구비율_취약점수']

# 엔트로피 계산 전 필수 점검: 여기서 하나라도 0이 아니면, entropy_weights의
# dropna()가 그 행들을 통째로 날려서 가중치 자체가 왜곡될 수 있음 -> 반드시 확인
print('엔트로피 계산 직전 변수별 결측 개수:')
print(eupmyeon_master[final_vars].isna().sum())

def entropy_weights(df, cols, eps=1e-6):
    X = df[cols].copy().dropna()
    X = X * (1 - 2*eps) + eps
    P = X / X.sum(axis=0)
    n = len(X)
    E = -(P * np.log(P)).sum(axis=0) / np.log(n)
    D = 1 - E
    W = D / D.sum()
    return W

weights = entropy_weights(eupmyeon_master, final_vars)
print(weights)

eupmyeon_master['이동격차지수'] = sum(eupmyeon_master[v].fillna(0) * weights[v] for v in final_vars)

final_ranked = eupmyeon_master[~eupmyeon_master['sidonm'].str.contains('서울')].copy()
final_ranked = final_ranked.sort_values('이동격차지수', ascending=False)
print(final_ranked[['읍면_key'] + final_vars + ['이동격차지수']].head(20))

엔트로피 계산 직전 변수별 결측 개수:
지형장벽_취약점수       0
자가용미보유_취약점수    77
의료_취약점수         0
버스정류장_취약점수      0
배차간격_취약점수       0
고령인구비율_취약점수    85
dtype: int64
지형장벽_취약점수      0.160038
자가용미보유_취약점수    0.005189
의료_취약점수        0.336894
버스정류장_취약점수     0.010236
배차간격_취약점수      0.399321
고령인구비율_취약점수    0.088322
dtype: float64
               읍면_key  지형장벽_취약점수  자가용미보유_취약점수   의료_취약점수  버스정류장_취약점수  \
755      경상남도 하동군 북천면   0.512814     0.878788  0.143456    0.740569   
79       경상북도 영양군 수비면   0.770311     0.848485  0.292192    0.528027   
1104     경상남도 밀양시 초동면   0.430421     0.878788  0.148001    0.919073   
122      경상북도 성주군 가천면   0.734061     0.818182  0.273239    0.838599   
119      경상북도 영양군 청기면   0.734749     0.848485  0.302812    0.463525   
103   강원특별자치도 강릉시 왕산면   0.745806     0.878788  0.179558    0.527902   
62       충청북도 영동군 용화면   0.793058     0.878788  0.251777    0.475824   
760      경상북도 봉화군 상운면   0.511653     0.848485  0.119088    0.705873   
192   강원특별자치도 평창군 봉평면   0.690805     0.848485  0.310582    

**## 6. 동일가중치와 비교 (민감도분석)**

In [9]:
equal_w = {v: 1/len(final_vars) for v in final_vars}
eupmyeon_master['이동격차지수_동일가중'] = sum(
    eupmyeon_master[v].fillna(0) * equal_w[v] for v in final_vars
)

comparison = eupmyeon_master[['읍면_key']].copy()
comparison['순위_엔트로피'] = eupmyeon_master['이동격차지수'].rank(ascending=False)
comparison['순위_동일가중'] = eupmyeon_master['이동격차지수_동일가중'].rank(ascending=False)
comparison['순위차이'] = (comparison['순위_엔트로피'] - comparison['순위_동일가중']).abs()

print('상위 20위 안에서 순위가 가장 많이 바뀐 지역:')
print(comparison.sort_values('순위차이', ascending=False).head(20))

상위 20위 안에서 순위가 가장 많이 바뀐 지역:
                 읍면_key  순위_엔트로피  순위_동일가중    순위차이
2305  전남광주통합특별시 신안군 장산면    649.0   3403.0  2754.0
1854  전남광주통합특별시 신안군 임자면    816.0   3395.0  2579.0
2224  전남광주통합특별시 신안군 비금면    919.0   3443.0  2524.0
2084  전남광주통합특별시 신안군 지도읍    929.0   3446.0  2517.0
2784  전남광주통합특별시 신안군 증도면    988.0   3460.0  2472.0
1382  전남광주통합특별시 신안군 암태면   1044.0   3458.0  2414.0
1379  전남광주통합특별시 신안군 도초면    932.0   3337.0  2405.0
2547  전남광주통합특별시 신안군 안좌면   1098.0   3475.0  2377.0
1690  전남광주통합특별시 신안군 신의면   1136.0   3473.0  2337.0
1958  전남광주통합특별시 신안군 자은면   1179.0   3477.0  2298.0
1871  전남광주통합특별시 신안군 하의면    743.0   2989.0  2246.0
2633  전남광주통합특별시 신안군 팔금면   1366.0   3476.0  2110.0
3407  전남광주통합특별시 해남군 산이면   1331.0   3406.0  2075.0
1825      인천광역시 옹진군 영흥면   1234.0   3245.0  2011.0
2174  전남광주통합특별시 신안군 압해읍   1476.0   3478.0  2002.0
2435     경기도 화성시만세구 송산면   1560.0   3481.0  1921.0
1777  전남광주통합특별시 해남군 해남읍   1346.0   3251.0  1905.0
2497  전남광주통합특별시 해남군 황산면   1091.0   2969.0  1878.0
2940     경기도 화성시만세구 우정

**## 7. 아직 남은 것**

- 공급공백: 마을조사표 기반 지표, 아직 미완성
- 의료를 HIRA 버전(0.4/0.6)으로 바꿨으므로 5·6번을 다시 돌려야 한다. 동 지역 0 대체가 사라지면 의료_취약점수의 엔트로피 가중치(기존 58.9%)와 엔트로피 vs 동일가중치 순위 차이(신안군·해남군 쏠림)가 크게 달라질 수 있어서, 이전 1차·2차 결과 정리본의 의료 관련 해석은 새 결과로 대체해야 한다.
- 은규 데이터(지형장벽/의료)의 `sidonm`/`sggnm` 값이 2026.7.1 통합 이전 표기(광주광역시/전라남도)인지 이후 표기(전남광주통합특별시)인지 확인 — `resolve_sido()`가 양쪽 다 같은 키로 수렴시켜주니 코드 자체는 그대로 둬도 되지만, 결측 체크에서 이상 패턴이 보이면 이 부분부터 의심할 것
- 버스(정류장·배차간격) 146개 읍면동은 가연의 인구 원본에 시군구 정보가 없어 생기는 구조적 한계(동명 중복 136개, 미대응 10개)로, 4번에서 시군구→시도 평균으로 잠정 대체함. 가연이 시군구 포함된 더 나은 인구 데이터를 구하면 그걸로 다시 매칭해서 대체 처리를 없애는 게 이상적 — 최종 보고서에는 "N개 읍면동은 지역 평균으로 대체"라고 명시할 것
- 위 항목들 완성되면 `final_vars` 리스트에 추가하고 그대로 다시 돌리면 됨

## 8. 최초 이름 일치 비교 (참고용 기준선)

최신 판정은 아래 9번의 경계 기준·후보 관리 결과를 사용합니다. 8번은 이전 이름 비교를 보존한 기준선입니다.

앞의 최종 지수 계산을 실행한 다음 실행합니다. 기존 Colab에 계산 결과가 남아 있으면 아래 코드 셀만 복사해도 됩니다.
전국 기준 가중치와 점수를 그대로 유지하고 경기도만 추출합니다. 경기도만으로 가중치를 다시 계산하지 않습니다.

제공된 운행현황 CSV 사본을 코드에 포함했습니다. 현재 시점의 운행을 별도로 확인한 자료가 아닙니다.
시군과 읍면동 명칭이 운행구역에 명시된 경우만 확인으로 표시합니다. 신도시·지구명, 법정동/행정동 차이와 개편 명칭은 자동 추정하지 않습니다.
‘미확인’ 또는 ‘기록 없음’은 미운행 판정이 아닙니다. 명칭이 일치해도 읍면동 전역을 운행한다는 뜻은 아닙니다.
이 자료에는 승객·탑승 기록이 없어 주민의 실제 이용 여부나 이용률을 판단할 수 없습니다.

결측 포함 지역은 잠정 순위에 주의 표시를 붙입니다. 결측 없는 지역만의 순위도 함께 제공하지만 기존 가중치는 그대로이므로 데이터 문제를 모두 해결한 결과는 아닙니다.
상위 10/20/50은 점수 순 정렬 후 해당 행 수를 선택합니다(경계 동점의 전원 포함 방식이 아님).


In [ ]:
"""최종 노트북의 eupmyeon_master를 받아 경기도와 똑버스 CSV를 비교한다."""
import re
from pathlib import Path
import pandas as pd


def compare_gyeonggi_drt(master, drt, output_dir='gyeonggi_drt_comparison'):
    required = ['sidonm', 'sggnm', 'emdnm', '이동격차지수']
    missing = set(required) - set(master.columns)
    if missing:
        raise ValueError(f'최종 지수 계산을 먼저 실행하세요. 누락 열: {missing}')
    gg = master.loc[master['sidonm'].eq('경기도')].copy()
    if gg.empty:
        raise ValueError('경기도 행이 없습니다.')
    # 전국에서 계산된 점수와 가중치를 유지한 채 경기도만 추출한다.
    gg['비교_시군'] = gg['sggnm'].str.extract(r'^(.+?[시군])', expand=False)
    gg['경기도순위_잠정'] = gg['이동격차지수'].rank(method='min', ascending=False).astype('Int64')
    score_cols = [c for c in master.columns if c.endswith('_취약점수')]
    gg['결측항목'] = gg[score_cols].apply(lambda row: ', '.join(row.index[row.isna()]), axis=1)
    gg['순위주의'] = gg['결측항목'].ne('')
    gg['결측없는지역내순위'] = gg.loc[~gg['순위주의'], '이동격차지수'].rank(method='min', ascending=False).astype('Int64')
    drt = drt.loc[drt['지자체'].ne('계')].copy()
    drt['비교_시군'] = drt['지자체'].str.extract(r'^(.+?[시군])', expand=False)
    # 원문에 읍/면 접미사가 생략된 두 표현만 명시적으로 확장한다.
    expansions = {'은현·남면·덕정역': '은현면·남면·덕정역', '남사·이동읍': '남사읍·이동읍',
                  '광교1·2동': '광교1동·광교2동'}
    def tokens(area):
        area = expansions.get(str(area), str(area))
        return set(re.findall(r'[가-힣0-9]+(?:읍|면|동)(?=$|[\s,·()])', area))
    drt['명시지역'] = drt['운행구역'].map(tokens)
    def match(row):
        city_rows = drt.loc[drt['비교_시군'].eq(row['비교_시군'])]
        hits = city_rows.loc[city_rows['명시지역'].map(lambda x: row['emdnm'] in x)]
        if len(hits):
            status = '운행구역에 읍면동 명시'
        elif len(city_rows):
            status = '읍면동 미확인(해당 시군 운행 기록 있음)'
        else:
            status = '제공 CSV에 해당 시군 기록 없음'
        return pd.Series({
            'DRT대조결과': status,
            '일치_운행구역': ' / '.join(hits['운행구역'].astype(str)),
            '일치_운행개시일': ' / '.join(hits['운행개시일'].astype(str)),
            '시군전체_운행구역': ' / '.join(city_rows['운행구역'].astype(str)),
        })
    gg = pd.concat([gg, gg.apply(match, axis=1)], axis=1)
    gg = gg.sort_values('이동격차지수', ascending=False)
    front = ['경기도순위_잠정', 'sggnm', 'emdnm', '이동격차지수', 'DRT대조결과',
             '일치_운행구역', '일치_운행개시일', '순위주의', '결측항목', '결측없는지역내순위']
    gg = gg[front + [c for c in gg.columns if c not in front]]
    summaries = []
    for label, subset in [('전체', gg), ('결측없는지역', gg.loc[~gg['순위주의']])]:
        for k in (10, 20, 50):
            top = subset.head(k)
            confirmed = top['DRT대조결과'].eq('운행구역에 읍면동 명시').sum()
            summaries.append({'대상': label, '상위N': k, '실제지역수': len(top),
                              '읍면동명시운행지역수': int(confirmed),
                              '명시확인비율': confirmed / len(top) if len(top) else float('nan'),
                              '결측포함지역수': int(top['순위주의'].sum())})
    summary = pd.DataFrame(summaries)
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    gg.to_csv(output / '경기도_이동격차순위_DRT대조.csv', index=False, encoding='utf-8-sig')
    gg.head(20).to_csv(output / '경기도_상위20_DRT대조.csv', index=False, encoding='utf-8-sig')
    summary.to_csv(output / '경기도_상위권_DRT요약.csv', index=False, encoding='utf-8-sig')
    return gg, summary


from io import StringIO
drt_source = pd.read_csv(StringIO('지자체,운행구역,운행대수,운행개시일,추진근거\n계,19개 市,267대,,\n파주시 (24대),운정신도시·교하지구,15대,2021년 12월,규제샌드박스\n파주시 (24대),광탄면,3대,2023년 12월,농어촌\n파주시 (24대),탄현면,3대,2023년 12월,농어촌\n파주시 (24대),운정4동,3대,2023년 12월,대중교통취약지구\n안산시 (16대),대부도,6대,2023년 3월,규제샌드박스\n안산시 (16대),스마트허브,10대,2024년 7월,규제샌드박스\n평택시,고덕국제신도시,15대,2023년 4월,규제샌드박스\n수원시 (30대),광교1·2동,10대,2023년 5월,규제샌드박스\n수원시 (30대),고색동(델타플렉스),10대,2024년 10월,규제샌드박스\n수원시 (30대),당수동,10대,2025년 6월,규제샌드박스\n고양시 (14대),고봉동,3대,2023년 6월,규제샌드박스\n고양시 (14대),식사동,4대,2023년 6월,규제샌드박스\n고양시 (14대),향동동,4대,2024년 9월,규제샌드박스\n고양시 (14대),덕은동,3대,2024년 11월,규제샌드박스\n화성시 (35대),동탄신도시,20대,2023년 6~7월,규제샌드박스\n화성시 (35대),향남1·2지구,5대,2023년 6월,규제샌드박스\n화성시 (35대),봉담읍,5대,2024년 9월,농어촌\n화성시 (35대),남양읍·마도면,5대,2024년 9월,농어촌\n김포시,"고촌읍, 풍무동",10대,2023년 6월,규제샌드박스\n양주시 (25대),"옥정신도시, 삼숭동, 회천지구",15대,2023년 7월,규제샌드박스\n양주시 (25대),백석읍·광적면·양주역,4대,2025년 6월,농어촌\n양주시 (25대),은현·남면·덕정역,3대,2025년 6월,농어촌\n양주시 (25대),장흥면,3대,2025년 6월,농어촌\n하남시 (9대),감일동,5대,2023년 8월,규제샌드박스\n하남시 (9대),위례동,4대,2023년 8월,규제샌드박스\n안성시 (15대),"서부권(공도읍,양성면,원곡면)",4대,2023년 10월,농어촌\n안성시 (15대),"동부권(일죽면,죽산면,삼죽면)",4대,2023년 10월,농어촌\n안성시 (15대),"북부권(보개면,대덕면,고삼면,양성면)",4대,2024년 5월,농어촌\n안성시 (15대),"남부권(금광면,서운면,미양면,보개면)",3대,2024년 5월,농어촌\n이천시 (22대),"시내권(관고동,중리동,증포동,창전동)",12대,2023년 12월,규제샌드박스\n이천시 (22대),장호원읍,7대,2023년 12월,농어촌\n이천시 (22대),율면,3대,2023년 12월,농어촌\n부천시 (7대),"고강본동, 고강1동",3대,2024년 4월,규제샌드박스\n부천시 (7대),"옥길동, 범박동",4대,2024년 4월,규제샌드박스\n광주시 (15대),곤지암읍,4대,2024년 5월,농어촌\n광주시 (15대),도척면,4대,2024년 5월,농어촌\n광주시 (15대),초월읍,4대,2024년 6월,농어촌\n광주시 (15대),퇴촌면,3대,2024년 6월,농어촌\n여주시 (10대),세종대왕면·중앙동·오학동,8대,2024년 6월,농어촌\n여주시 (10대),북내면·강천면,2대,2024년 6월,농어촌\n안양시,안양9동,2대,2024년 8월,규제샌드박스\n연천군,"연천읍, 내산리",2대,2024년 8월,농어촌\n의왕시,초평동,2대,2025년 6월,규제샌드박스\n용인시,남사·이동읍,6대,2025년 6월,농어촌\n의정부시,민락1지구,4대,2025년 6월,규제샌드박스\n의정부시,민락2·고산지구,4대,2025년 6월,규제샌드박스\n'))

gyeonggi_drt, gyeonggi_drt_summary = compare_gyeonggi_drt(eupmyeon_master, drt_source)
display(gyeonggi_drt.head(20)[['경기도순위_잠정', 'sggnm', 'emdnm', '이동격차지수', 'DRT대조결과', '일치_운행구역', '순위주의']])
display(gyeonggi_drt_summary)


In [ ]:
# 비교 결과 CSV 3개를 ZIP으로 내려받기 (Colab)
import shutil
shutil.make_archive('gyeonggi_drt_comparison', 'zip', 'gyeonggi_drt_comparison')
from google.colab import files
files.download('gyeonggi_drt_comparison.zip')


## 9. 경계 기준 비교 — 9개 권역 후보 관리
2026-07-01 경계 코드에 연결한 대응표를 포함했습니다. 9개 권역은 모두 후보로 집계하며 원래 확인 근거는 별도 보존합니다. 후보를 운행 확인 수에 포함하지 않습니다. 다른 권역에서 확인된 동일 읍면동은 확인으로 남고 후보연결존재도 표시합니다.
차량수는 운행권역표에 권역 합계로 보존합니다. 읍면동별 배분값은 없습니다. 양주역·덕정역은 연계거점으로만 남깁니다. 대응표는 작성 시점의 고정 자료이며 지수만 현재 실행 결과로 연결합니다. 원본·경계·정책 변경 시 로컬 변환 및 노트북 생성 스크립트를 다시 실행하세요.


In [ ]:
from io import StringIO
boundary_links = pd.read_csv(StringIO('읍면_key,emd8,emdcd,경계기준_DRT연결,확인_운행구역ID,후보_운행구역ID,후보연결존재,연결_원본운행구역,경계변환_근거URL\r\n경기도 여주시 산북면,31280350,4167038000,미확인,,,False,,\r\n경기도 연천군 중면,31550370,4180037000,미확인,,,False,,\r\n경기도 여주시 북내면,31280370,4167036000,확인,DRT040,,False,북내면·강천면,\r\n경기도 양평군 양동면,31580380,4183038000,미확인,,,False,,\r\n경기도 가평군 북면,31570350,4182035000,미확인,,,False,,\r\n경기도 양평군 옥천면,31580340,4183034000,미확인,,,False,,\r\n경기도 연천군 왕징면,31550350,4180035000,미확인,,,False,,\r\n경기도 이천시 모가면,31210360,4150036000,미확인,,,False,,\r\n경기도 여주시 강천면,31280380,4167037000,확인,DRT040,,False,북내면·강천면,\r\n경기도 양평군 단월면,31580360,4183036000,미확인,,,False,,\r\n경기도 이천시 율면,31210380,4150038000,확인,DRT032,,False,율면,\r\n경기도 양평군 서종면,31580350,4183035000,미확인,,,False,,\r\n경기도 양평군 청운면,31580370,4183037000,미확인,,,False,,\r\n경기도 포천시 관인면,31270400,4165040000,미확인,,,False,,\r\n경기도 안성시 삼죽면,31220410,4155041000,확인,DRT027,,False,"동부권(일죽면,죽산면,삼죽면)",\r\n경기도 가평군 가평읍,31570110,4182025000,미확인,,,False,,\r\n경기도 연천군 신서면,31550360,4180036000,확인,DRT042,,False,"연천읍, 내산리",https://yeoncheon.go.kr/www/contents.do?key=3410 | https://www.yeoncheon.go.kr/www/contents.do?key=3441\r\n경기도 연천군 장남면,31550380,4180038000,미확인,,,False,,\r\n경기도 가평군 설악면,31570310,4182031000,미확인,,,False,,\r\n경기도 안성시 죽산면,31220400,4155040000,확인,DRT027,,False,"동부권(일죽면,죽산면,삼죽면)",\r\n경기도 가평군 조종면,31570360,4182034500,미확인,,,False,,\r\n경기도 양평군 지평면,31580390,4183039500,미확인,,,False,,\r\n경기도 양평군 용문면,31580400,4183040000,미확인,,,False,,\r\n경기도 동두천시 소요동,31080580,4125058000,미확인,,,False,,\r\n경기도 가평군 상면,31570330,4182033000,미확인,,,False,,\r\n경기도 안성시 양성면,31220360,4155036000,확인,DRT026 / DRT028,,False,"서부권(공도읍,양성면,원곡면) / 북부권(보개면,대덕면,고삼면,양성면)",\r\n경기도 연천군 백학면,31550330,4180033000,미확인,,,False,,\r\n경기도 양평군 개군면,31580410,4183041000,미확인,,,False,,\r\n경기도 가평군 청평면,31570320,4182032500,미확인,,,False,,\r\n경기도 양평군 강하면,31580320,4183032000,미확인,,,False,,\r\n경기도 양평군 양서면,31580330,4183033000,미확인,,,False,,\r\n경기도 안성시 고삼면,31220420,4155042000,확인,DRT028,,False,"북부권(보개면,대덕면,고삼면,양성면)",\r\n경기도 양평군 강상면,31580310,4183031000,미확인,,,False,,\r\n경기도 여주시 점동면,31280310,4167031000,미확인,,,False,,\r\n경기도 여주시 세종대왕면,31280390,4167034500,확인,DRT039,,False,세종대왕면·중앙동·오학동,\r\n경기도 이천시 대월면,31210350,4150035000,미확인,,,False,,\r\n경기도 연천군 연천읍,31550110,4180025000,확인,DRT042,,False,"연천읍, 내산리",\r\n경기도 안성시 보개면,31220310,4155031000,확인,DRT028 / DRT029,,False,"북부권(보개면,대덕면,고삼면,양성면) / 남부권(금광면,서운면,미양면,보개면)",\r\n경기도 안성시 일죽면,31220390,4155039000,확인,DRT027,,False,"동부권(일죽면,죽산면,삼죽면)",\r\n경기도 안성시 금광면,31220320,4155032000,확인,DRT029,,False,"남부권(금광면,서운면,미양면,보개면)",\r\n경기도 포천시 이동면,31270380,4165038000,미확인,,,False,,\r\n경기도 광주시 남한산성면,31250380,4161037000,미확인,,,False,,\r\n경기도 포천시 창수면,31270350,4165035000,미확인,,,False,,\r\n경기도 광주시 남종면,31250360,4161035000,미확인,,,False,,\r\n경기도 연천군 청산면,31550320,4180032000,미확인,,,False,,\r\n경기도 이천시 장호원읍,31210110,4150025000,확인,DRT031,,False,장호원읍,\r\n경기도 여주시 금사면,31280340,4167033000,미확인,,,False,,\r\n경기도 포천시 신북면,31270340,4165034000,미확인,,,False,,\r\n경기도 연천군 군남면,31550310,4180031000,미확인,,,False,,\r\n경기도 여주시 흥천면,31280330,4167032000,미확인,,,False,,\r\n경기도 안성시 원곡면,31220380,4155038000,확인,DRT026,,False,"서부권(공도읍,양성면,원곡면)",\r\n경기도 여주시 대신면,31280360,4167035000,미확인,,,False,,\r\n경기도 포천시 일동면,31270370,4165037000,미확인,,,False,,\r\n경기도 연천군 미산면,31550340,4180034000,미확인,,,False,,\r\n경기도 이천시 호법면,31210330,4150033000,미확인,,,False,,\r\n경기도 남양주시 수동면,31130340,4136034000,미확인,,,False,,\r\n경기도 포천시 영북면,31270390,4165039000,미확인,,,False,,\r\n경기도 하남시 천현동,31180510,4145051000,미확인,,,False,,\r\n경기도 동두천시 불현동,31080560,4125056500,미확인,,,False,,\r\n경기도 안성시 서운면,31220330,4155033000,확인,DRT029,,False,"남부권(금광면,서운면,미양면,보개면)",\r\n경기도 광주시 곤지암읍,31250140,4161025900,확인,DRT035,,False,곤지암읍,\r\n경기도 용인시처인구 동부동,31191540,4146154000,미확인,,,False,,\r\n경기도 용인시처인구 백암면,31191350,4146135000,미확인,,,False,,\r\n경기도 포천시 영중면,31270360,4165036000,미확인,,,False,,\r\n경기도 용인시처인구 이동읍,31191130,4146125600,확인,DRT044,,False,남사·이동읍,\r\n경기도 이천시 설성면,31210370,4150037000,미확인,,,False,,\r\n경기도 파주시 법원읍,31200130,4148025600,미확인,,,False,,\r\n경기도 이천시 신둔면,31210310,4150031000,미확인,,,False,,\r\n경기도 이천시 백사면,31210320,4150032000,미확인,,,False,,\r\n경기도 여주시 가남읍,31280110,4167025000,미확인,,,False,,\r\n경기도 동두천시 보산동,31080550,4125055000,미확인,,,False,,\r\n경기도 남양주시 조안면,31130350,4136036000,미확인,,,False,,\r\n경기도 포천시 군내면,31270310,4165031000,미확인,,,False,,\r\n경기도 파주시 장단면,31200390,4148039000,미확인,,,False,,\r\n경기도 포천시 내촌면,31270320,4165032000,미확인,,,False,,\r\n경기도 광주시 도척면,31250340,4161033000,확인,DRT036,,False,도척면,\r\n경기도 파주시 파평면,31200360,4148036000,미확인,,,False,,\r\n경기도 양평군 양평읍,31580110,4183025000,미확인,,,False,,\r\n경기도 하남시 춘궁동,31180590,4145059000,미확인,,,False,,\r\n경기도 안산시단원구 대부동,31092720,4127361000,확인,DRT005,,False,대부도,https://www.ansan.go.kr/stat/common/cntnts/selectContents.do?cntnts_id=C0001013 | https://www.ansan.go.kr/danwongu/common/cntnts/selectContents.do?cntnts_id=C0000880\r\n경기도 광주시 퇴촌면,31250350,4161034000,확인,DRT038,,False,퇴촌면,\r\n경기도 여주시 여흥동,31280510,4167051000,미확인,,,False,,\r\n경기도 이천시 마장면,31210340,4150034000,미확인,,,False,,\r\n경기도 연천군 전곡읍,31550120,4180025300,미확인,,,False,,\r\n경기도 양주시 장흥면,31260340,4163034000,확인,DRT023,,False,장흥면,\r\n경기도 파주시 적성면,31200370,4148037000,미확인,,,False,,\r\n경기도 남양주시 별내면,31130310,4136031000,미확인,,,False,,\r\n경기도 양주시 광적면,31260330,4163033000,후보,,DRT021,True,백석읍·광적면·양주역,\r\n경기도 안성시 대덕면,31220350,4155035000,확인,DRT028,,False,"북부권(보개면,대덕면,고삼면,양성면)",\r\n경기도 광주시 초월읍,31250120,4161025300,확인,DRT037,,False,초월읍,\r\n경기도 파주시 광탄면,31200350,4148035000,확인,DRT002,,False,광탄면,\r\n경기도 포천시 화현면,31270410,4165041000,미확인,,,False,,\r\n경기도 남양주시 오남읍,31130150,4136026200,미확인,,,False,,\r\n경기도 양주시 은현면,31260310,4163031000,후보,,DRT022,True,은현·남면·덕정역,\r\n경기도 용인시처인구 원삼면,31191340,4146134000,미확인,,,False,,\r\n경기도 이천시 관고동,31210530,4150053000,확인,DRT030,,False,"시내권(관고동,중리동,증포동,창전동)",\r\n경기도 고양시덕양구 효자동,31101560,4128156000,미확인,,,False,,\r\n경기도 여주시 오학동,31280530,4167053000,확인,DRT039,,False,세종대왕면·중앙동·오학동,\r\n경기도 안양시만안구 병목안동,31041672,4117158300,확인,DRT041,,False,안양9동,https://www.anyang.go.kr/main/contents.do?key=289\r\n경기도 동두천시 상패동,31080600,4125060000,미확인,,,False,,\r\n경기도 남양주시 화도읍,31130130,4136025600,미확인,,,False,,\r\n경기도 용인시처인구 양지읍,31191581,4146126200,미확인,,,False,,\r\n경기도 과천시 문원동,31110560,4129056000,미확인,,,False,,\r\n경기도 이천시 중리동,31210520,4150052000,확인,DRT030,,False,"시내권(관고동,중리동,증포동,창전동)",\r\n경기도 안성시 미양면,31220340,4155034000,확인,DRT029,,False,"남부권(금광면,서운면,미양면,보개면)",\r\n경기도 양주시 양주1동,31260510,4163051000,미확인,,,False,,\r\n경기도 이천시 부발읍,31210120,4150025300,미확인,,,False,,\r\n경기도 남양주시 와부읍,31130110,4136025000,미확인,,,False,,\r\n경기도 이천시 창전동,31210510,4150051000,확인,DRT030,,False,"시내권(관고동,중리동,증포동,창전동)",\r\n경기도 군포시 대야동,31160630,4141061000,미확인,,,False,,\r\n경기도 포천시 가산면,31270330,4165033000,미확인,,,False,,\r\n경기도 동두천시 생연1동,31080510,4125051000,미확인,,,False,,\r\n경기도 화성시만세구 송산면,31241725,4159132000,미확인,,,False,,\r\n경기도 포천시 소흘읍,31270110,4165025000,미확인,,,False,,\r\n경기도 과천시 중앙동,31110510,4129051000,미확인,,,False,,\r\n경기도 김포시 월곶면,31230350,4157035000,미확인,,,False,,\r\n경기도 광주시 탄벌동,31250570,4161055000,미확인,,,False,,\r\n경기도 의정부시 장암동,31030560,4115056100,미확인,,,False,,\r\n경기도 안양시동안구 비산3동,31042530,4117353000,미확인,,,False,,\r\n경기도 광주시 광남1동,31250580,4161056000,미확인,,,False,,\r\n경기도 양주시 백석읍,31260110,4163025000,후보,,DRT021,True,백석읍·광적면·양주역,\r\n경기도 성남시중원구 은행2동,31022560,4113356000,미확인,,,False,,\r\n경기도 용인시처인구 남사읍,31191140,4146125900,확인,DRT044,,False,남사·이동읍,\r\n경기도 용인시처인구 모현읍,31191120,4146125300,미확인,,,False,,\r\n경기도 양주시 남면,31260320,4163032000,후보,,DRT022,True,은현·남면·덕정역,\r\n경기도 의정부시 호원2동,31030650,4115055500,미확인,,,False,,\r\n경기도 평택시 진위면,31070310,4122031000,미확인,,,False,,\r\n경기도 평택시 송북동,31070560,4122056000,미확인,,,False,,\r\n경기도 안산시상록구 안산동,31091730,4127161000,미확인,,,False,,\r\n경기도 안양시만안구 석수1동,31041600,4117159000,미확인,,,False,,\r\n경기도 포천시 선단동,31270520,4165052000,미확인,,,False,,\r\n경기도 광명시 학온동,31060680,4121066000,미확인,,,False,,\r\n경기도 수원시장안구 연무동,31011610,4111160000,미확인,,,False,,\r\n경기도 용인시처인구 삼가동,31191560,4146152600,미확인,,,False,,\r\n경기도 평택시 오성면,31070340,4122034000,미확인,,,False,,\r\n경기도 고양시덕양구 고양동,31101590,4128159000,미확인,,,False,,\r\n경기도 용인시처인구 포곡읍,31191110,4146125000,미확인,,,False,,\r\n경기도 남양주시 호평동,31130510,4136051000,미확인,,,False,,\r\n경기도 고양시일산동구 고봉동,31103600,4128560000,확인,DRT011,,False,고봉동,\r\n경기도 의정부시 흥선동,31030670,4115061500,미확인,,,False,,\r\n경기도 안성시 공도읍,31220110,4155025000,확인,DRT026,,False,"서부권(공도읍,양성면,원곡면)",\r\n경기도 용인시수지구 동천동,31193560,4146556000,미확인,,,False,,\r\n경기도 화성시만세구 우정읍,31241721,4159125000,미확인,,,False,,\r\n경기도 평택시 현덕면,31070370,4122037000,미확인,,,False,,\r\n경기도 포천시 포천동,31270510,4165051000,미확인,,,False,,\r\n경기도 광주시 오포1동,31250511,4161058000,미확인,,,False,,\r\n경기도 화성시만세구 장안면,31241728,4159135000,미확인,,,False,,\r\n경기도 성남시분당구 운중동,31023680,4113568000,미확인,,,False,,\r\n경기도 성남시수정구 양지동,31021620,4113161000,미확인,,,False,,\r\n경기도 남양주시 평내동,31130520,4136052000,미확인,,,False,,\r\n경기도 의정부시 호원1동,31030550,4115054500,미확인,,,False,,\r\n경기도 동두천시 송내동,31080610,4125056600,미확인,,,False,,\r\n경기도 의왕시 오전동,31170530,4143053000,미확인,,,False,,\r\n경기도 파주시 교하동,31200511,4148054000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 구리시 교문1동,31120540,4131054100,미확인,,,False,,\r\n경기도 양주시 회천1동,31260530,4163053000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",\r\n경기도 용인시기흥구 보라동,31192660,4146353500,미확인,,,False,,\r\n경기도 화성시만세구 마도면,31241724,4159131000,확인,DRT018,,False,남양읍·마도면,\r\n경기도 용인시처인구 유림1동,31191570,4146153500,미확인,,,False,,\r\n경기도 김포시 하성면,31230360,4157036000,미확인,,,False,,\r\n경기도 남양주시 금곡동,31130530,4136053000,미확인,,,False,,\r\n경기도 남양주시 양정동,31130540,4136054000,미확인,,,False,,\r\n경기도 화성시만세구 팔탄면,31241727,4159134000,미확인,,,False,,\r\n경기도 군포시 수리동,31160590,4141059000,미확인,,,False,,\r\n경기도 성남시중원구 상대원1동,31022570,4113357000,미확인,,,False,,\r\n경기도 용인시수지구 신봉동,31193530,4146553000,미확인,,,False,,\r\n경기도 파주시 금촌3동,31200540,4148053000,미확인,,,False,,\r\n경기도 광주시 신현동,31250513,4161060000,미확인,,,False,,\r\n경기도 파주시 조리읍,31200150,4148026200,미확인,,,False,,\r\n경기도 의왕시 청계동,31170560,4143056000,미확인,,,False,,\r\n경기도 성남시수정구 시흥동,31021660,4113165000,미확인,,,False,,\r\n경기도 양주시 회천2동,31260540,4163054000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",\r\n경기도 하남시 초이동,31180600,4145060000,미확인,,,False,,\r\n경기도 화성시효행구 매송면,31242732,4159331000,미확인,,,False,,\r\n경기도 안산시상록구 월피동,31091580,4127158000,미확인,,,False,,\r\n경기도 고양시덕양구 관산동,31101600,4128160000,미확인,,,False,,\r\n경기도 남양주시 진접읍,31130120,4136025300,미확인,,,False,,\r\n경기도 군포시 궁내동,31160600,4141060000,미확인,,,False,,\r\n경기도 이천시 증포동,31210540,4150051500,확인,DRT030,,False,"시내권(관고동,중리동,증포동,창전동)",\r\n경기도 의왕시 고천동,31170510,4143051000,미확인,,,False,,\r\n경기도 광주시 오포2동,31250512,4161059000,미확인,,,False,,\r\n경기도 화성시만세구 서신면,31241726,4159133000,미확인,,,False,,\r\n경기도 김포시 대곶면,31230340,4157034000,미확인,,,False,,\r\n경기도 하남시 감북동,31180660,4145058000,미확인,,,False,,\r\n경기도 과천시 과천동,31110550,4129055000,미확인,,,False,,\r\n경기도 오산시 세마동,31140550,4137055000,미확인,,,False,,\r\n경기도 안산시상록구 반월동,31091710,4127160000,미확인,,,False,,\r\n경기도 화성시만세구 양감면,31241729,4159136000,미확인,,,False,,\r\n경기도 광명시 철산4동,31060610,4121062000,미확인,,,False,,\r\n경기도 파주시 파주읍,31200120,4148025300,미확인,,,False,,\r\n경기도 화성시효행구 비봉면,31242733,4159332000,미확인,,,False,,\r\n경기도 과천시 원문동,31110522,4129051500,미확인,,,False,,\r\n경기도 안양시만안구 호현동,31041670,4117165000,미확인,,,False,,\r\n경기도 광주시 능평동,31250514,4161061000,미확인,,,False,,\r\n경기도 의왕시 내손1동,31170540,4143054000,미확인,,,False,,\r\n경기도 평택시 포승읍,31070130,4122025600,미확인,,,False,,\r\n경기도 안성시 안성2동,31220520,4155052000,미확인,,,False,,\r\n경기도 파주시 문산읍,31200110,4148025000,미확인,,,False,,\r\n경기도 용인시기흥구 상하동,31192600,4146358600,미확인,,,False,,\r\n경기도 광주시 쌍령동,31250560,4161054000,미확인,,,False,,\r\n경기도 광주시 송정동,31250550,4161052000,미확인,,,False,,\r\n경기도 의정부시 가능동,31030680,4115059500,미확인,,,False,,\r\n경기도 파주시 월롱면,31200310,4148031000,미확인,,,False,,\r\n경기도 수원시장안구 파장동,31011540,4111156000,미확인,,,False,,\r\n경기도 시흥시 과림동,31150590,4139062100,미확인,,,False,,\r\n경기도 평택시 서탄면,31070320,4122032000,미확인,,,False,,\r\n경기도 성남시분당구 금곡동,31023710,4113566200,미확인,,,False,,\r\n경기도 파주시 탄현면,31200320,4148032000,확인,DRT003,,False,탄현면,\r\n경기도 하남시 위례동,31180650,4145058500,확인,DRT025,,False,위례동,\r\n경기도 안양시만안구 안양2동,31041520,4117152000,미확인,,,False,,\r\n경기도 용인시기흥구 구성동,31192560,4146356000,미확인,,,False,,\r\n경기도 용인시기흥구 기흥동,31192540,4146354000,미확인,,,False,,\r\n경기도 화성시효행구 봉담읍,31242731,4159325000,확인,DRT017,,False,봉담읍,\r\n경기도 여주시 중앙동,31280520,4167052000,확인,DRT039,,False,세종대왕면·중앙동·오학동,\r\n경기도 용인시기흥구 마북동,31192570,4146357000,미확인,,,False,,\r\n경기도 성남시중원구 도촌동,31022610,4113367000,미확인,,,False,,\r\n경기도 남양주시 진건읍,31130140,4136025900,미확인,,,False,,\r\n경기도 안성시 안성1동,31220510,4155051000,미확인,,,False,,\r\n경기도 평택시 신장2동,31070580,4122058000,미확인,,,False,,\r\n경기도 김포시 통진읍,31230110,4157025000,미확인,,,False,,\r\n경기도 군포시 광정동,31160610,4141062000,미확인,,,False,,\r\n경기도 평택시 안중읍,31070120,4122025300,미확인,,,False,,\r\n경기도 안양시동안구 비산1동,31042510,4117351000,미확인,,,False,,\r\n경기도 성남시분당구 야탑3동,31023630,4113564000,미확인,,,False,,\r\n경기도 동두천시 중앙동,31080530,4125053500,미확인,,,False,,\r\n경기도 의정부시 자금동,31030600,4115058000,미확인,,,False,,\r\n경기도 고양시일산서구 가좌동,31104514,4128761000,미확인,,,False,,\r\n경기도 평택시 고덕면,31070330,4122033000,후보,,DRT007,True,고덕국제신도시,https://gnews.gg.go.kr/briefing/brief_gongbo_view.do?BS_CODE=S017&number=57039\r\n경기도 용인시처인구 중앙동,31191510,4146151000,미확인,,,False,,\r\n경기도 시흥시 매화동,31150550,4139055000,미확인,,,False,,\r\n경기도 광명시 하안1동,31060620,4121063100,미확인,,,False,,\r\n경기도 남양주시 별내동,31130570,4136057000,미확인,,,False,,\r\n경기도 성남시수정구 고등동,31021650,4113164000,미확인,,,False,,\r\n경기도 용인시기흥구 영덕2동,31192640,4146351700,미확인,,,False,,\r\n경기도 성남시분당구 구미1동,31023720,4113566500,미확인,,,False,,\r\n경기도 용인시기흥구 동백2동,31192680,4146357500,미확인,,,False,,\r\n경기도 안산시상록구 부곡동,31091570,4127157000,미확인,,,False,,\r\n경기도 의정부시 고산동,31030721,4115063000,후보,,DRT046,True,민락2·고산지구,https://www.ui4u.go.kr/cscportal/bbs/view.do?bIdx=348257&mId=0402000000&ptIdx=85\r\n경기도 평택시 팽성읍,31070110,4122025000,미확인,,,False,,\r\n경기도 안양시만안구 안양6동,31041560,4117156000,미확인,,,False,,\r\n경기도 용인시기흥구 동백1동,31192670,4146357200,미확인,,,False,,\r\n경기도 고양시덕양구 창릉동,31101580,4128158000,미확인,,,False,,\r\n경기도 성남시분당구 서현1동,31023580,4113558000,미확인,,,False,,\r\n경기도 부천시소사구 심곡본동,31052521,4119452000,미확인,,,False,,\r\n경기도 시흥시 목감동,31150560,4139057000,미확인,,,False,,\r\n경기도 성남시분당구 구미동,31023670,4113567000,미확인,,,False,,\r\n경기도 부천시원미구 춘의동,31051591,4119259000,미확인,,,False,,\r\n경기도 용인시수지구 성복동,31193590,4146559000,미확인,,,False,,\r\n경기도 안산시단원구 선부3동,31092700,4127360000,미확인,,,False,,\r\n경기도 의정부시 송산3동,31030710,4115057800,후보,,DRT046,True,민락2·고산지구,https://www.ui4u.go.kr/cscportal/bbs/view.do?bIdx=348257&mId=0402000000&ptIdx=85\r\n경기도 의정부시 송산1동,31030591,4115057300,후보,,DRT045,True,민락1지구,https://www.ui4u.go.kr/cscportal/songsan2/main.do?mId=0311010000\r\n경기도 안양시동안구 관양동,31042681,4117355200,미확인,,,False,,\r\n경기도 고양시덕양구 원신동,31101520,4128152000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄5동,31244745,4159755000,후보,,DRT015,True,동탄신도시,\r\n경기도 성남시수정구 태평2동,31021550,4113155000,미확인,,,False,,\r\n경기도 안산시상록구 성포동,31091620,4127159000,미확인,,,False,,\r\n경기도 양주시 양주2동,31260520,4163052000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",https://www.yangju.go.kr/www/contents.do?key=415\r\n경기도 하남시 감일동,31180670,4145058200,확인,DRT024,,False,감일동,\r\n경기도 광주시 광남2동,31250590,4161057000,미확인,,,False,,\r\n경기도 안산시단원구 초지동,31092660,4127357000,확인,DRT006,,False,스마트허브,https://smartcity.go.kr/wp-content/uploads/2026/02/안산시-스마트도시계획_26.2.pdf | https://www.ansan.go.kr/danwongu/common/cntnts/selectContents.do?cntnts_id=C0000914\r\n경기도 용인시처인구 역북동,31191550,4146152500,미확인,,,False,,\r\n경기도 하남시 덕풍1동,31180540,4145054000,미확인,,,False,,\r\n경기도 안성시 안성3동,31220530,4155053000,미확인,,,False,,\r\n경기도 의정부시 녹양동,31030640,4115062000,미확인,,,False,,\r\n경기도 광명시 소하2동,31060690,4121065000,미확인,,,False,,\r\n경기도 양주시 옥정1동,31260561,4163057000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",\r\n경기도 군포시 산본2동,31160550,4141055000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄9동,31244749,4159759000,후보,,DRT015,True,동탄신도시,\r\n경기도 성남시중원구 금광2동,31022540,4113354000,미확인,,,False,,\r\n경기도 시흥시 신현동,31150530,4139053100,미확인,,,False,,\r\n경기도 양주시 회천3동,31260550,4163055000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",\r\n경기도 안산시상록구 일동,31091510,4127151000,미확인,,,False,,\r\n경기도 고양시덕양구 화전동,31101670,4128166000,확인,DRT013,DRT014,True,향동동 / 덕은동,https://www.goyang.go.kr/dygu/dygu01/dygu01_5/dygu01_5_3.jsp\r\n경기도 화성시만세구 남양읍,31241723,4159125600,확인,DRT018,,False,남양읍·마도면,\r\n경기도 성남시수정구 단대동,31021600,4113159000,미확인,,,False,,\r\n경기도 용인시수지구 죽전3동,31193610,4146555500,미확인,,,False,,\r\n경기도 고양시덕양구 성사2동,31101550,4128155000,미확인,,,False,,\r\n경기도 용인시기흥구 신갈동,31192610,4146351000,미확인,,,False,,\r\n경기도 시흥시 군자동,31150680,4139058100,미확인,,,False,,\r\n경기도 평택시 송탄동,31070530,4122053500,미확인,,,False,,\r\n경기도 성남시분당구 이매1동,31023600,4113560000,미확인,,,False,,\r\n경기도 시흥시 월곶동,31150690,4139058200,미확인,,,False,,\r\n경기도 성남시중원구 하대원동,31022600,4113366000,미확인,,,False,,\r\n경기도 고양시일산동구 중산1동,31103511,4128552500,미확인,,,False,,\r\n경기도 안산시단원구 중앙동,31092750,4127353200,미확인,,,False,,\r\n경기도 구리시 인창동,31120530,4131053000,미확인,,,False,,\r\n경기도 용인시수지구 죽전1동,31193600,4146554000,미확인,,,False,,\r\n경기도 군포시 오금동,31160580,4141058000,미확인,,,False,,\r\n경기도 남양주시 퇴계원읍,31130160,4136026500,미확인,,,False,,\r\n경기도 파주시 금촌1동,31200530,4148051000,미확인,,,False,,\r\n경기도 성남시분당구 분당동,31023510,4113551000,미확인,,,False,,\r\n경기도 동두천시 생연2동,31080520,4125052000,미확인,,,False,,\r\n경기도 시흥시 대야동,31150510,4139051000,미확인,,,False,,\r\n경기도 고양시덕양구 주교동,31101510,4128151000,미확인,,,False,,\r\n경기도 안양시만안구 충훈동,31041651,4117161100,미확인,,,False,,\r\n경기도 성남시수정구 신촌동,31021640,4113163000,미확인,,,False,,\r\n경기도 용인시처인구 유림2동,31191580,4146153600,미확인,,,False,,\r\n경기도 평택시 청북읍,31070140,4122025900,미확인,,,False,,\r\n경기도 성남시중원구 은행1동,31022550,4113355000,미확인,,,False,,\r\n경기도 부천시소사구 심곡본1동,31052511,4119451000,미확인,,,False,,\r\n경기도 평택시 신평동,31070590,4122059000,미확인,,,False,,\r\n경기도 화성시만세구 향남읍,31241722,4159125300,확인,DRT016,,False,향남1·2지구,https://www.hscity.go.kr/www/user/bbs/BD_selectBbs.do?q_bbsCode=1010&q_bbscttSn=10100001315536\r\n경기도 성남시분당구 정자3동,31023560,4113557000,미확인,,,False,,\r\n경기도 하남시 덕풍2동,31180550,4145055000,미확인,,,False,,\r\n경기도 고양시덕양구 행주동,31101640,4128163000,미확인,,,False,,\r\n경기도 안산시상록구 사동,31091760,4127152500,미확인,,,False,,\r\n경기도 시흥시 능곡동,31150670,4139064000,미확인,,,False,,\r\n경기도 의정부시 신곡1동,31030570,4115056700,미확인,,,False,,\r\n경기도 안양시만안구 명학동,31041671,4117158200,미확인,,,False,,\r\n경기도 화성시효행구 정남면,31242734,4159333000,미확인,,,False,,\r\n경기도 용인시기흥구 보정동,31192590,4146359000,미확인,,,False,,\r\n경기도 안양시만안구 석수2동,31041610,4117160000,미확인,,,False,,\r\n경기도 수원시장안구 조원2동,31011630,4111159800,미확인,,,False,,\r\n경기도 군포시 재궁동,31160570,4141057000,미확인,,,False,,\r\n경기도 평택시 비전1동,31070670,4122062000,미확인,,,False,,\r\n경기도 오산시 신장2동,31140572,4137054600,미확인,,,False,,\r\n경기도 부천시원미구 역곡2동,31051581,4119258000,미확인,,,False,,\r\n경기도 의왕시 부곡동,31170520,4143052000,확인,DRT043,,False,초평동,https://www.uiwang.go.kr/UWKORINTRO0104\r\n경기도 성남시중원구 중앙동,31022620,4113352500,미확인,,,False,,\r\n경기도 부천시소사구 소사본동,31052551,4119453000,미확인,,,False,,\r\n경기도 성남시수정구 태평4동,31021570,4113156100,미확인,,,False,,\r\n경기도 김포시 양촌읍,31230130,4157025600,미확인,,,False,,\r\n경기도 부천시원미구 소사동,31051551,4119256000,미확인,,,False,,\r\n경기도 안양시만안구 안양3동,31041530,4117153000,미확인,,,False,,\r\n경기도 안산시단원구 고잔동,31092740,4127352500,미확인,,,False,,\r\n경기도 광명시 광명7동,31060570,4121058000,미확인,,,False,,\r\n경기도 고양시일산서구 일산1동,31104510,4128751000,미확인,,,False,,\r\n경기도 부천시원미구 역곡1동,31051571,4119257000,미확인,,,False,,\r\n경기도 성남시수정구 수진1동,31021580,4113157000,미확인,,,False,,\r\n경기도 평택시 신장1동,31070570,4122057000,미확인,,,False,,\r\n경기도 안산시단원구 신길동,31092770,4127356500,미확인,,,False,,\r\n경기도 안양시동안구 호계2동,31042630,4117359000,미확인,,,False,,\r\n경기도 구리시 동구동,31120520,4131052000,미확인,,,False,,\r\n경기도 성남시수정구 산성동,31021610,4113160000,미확인,,,False,,\r\n경기도 안산시단원구 와동,31092590,4127351000,미확인,,,False,,\r\n경기도 성남시분당구 판교동,31023750,4113565000,미확인,,,False,,\r\n경기도 성남시수정구 신흥1동,31021510,4113151000,미확인,,,False,,\r\n경기도 수원시팔달구 지동,31013530,4111570000,미확인,,,False,,\r\n경기도 시흥시 연성동,31150700,4139063000,미확인,,,False,,\r\n경기도 파주시 운정4동,31200514,4148058000,확인,DRT004,DRT001,True,운정신도시·교하지구 / 운정4동,\r\n경기도 성남시분당구 수내3동,31023520,4113554000,미확인,,,False,,\r\n경기도 용인시기흥구 동백3동,31192690,4146357700,미확인,,,False,,\r\n경기도 부천시오정구 성곡동,31053511,4119651000,미확인,,,False,,\r\n경기도 부천시원미구 원미1동,31051561,4119254000,미확인,,,False,,\r\n경기도 용인시기흥구 서농동,31192550,4146355000,미확인,,,False,,\r\n경기도 성남시수정구 신흥2동,31021520,4113152000,미확인,,,False,,\r\n경기도 고양시덕양구 삼송1동,31101512,4128157600,미확인,,,False,,\r\n경기도 수원시영통구 영통1동,31014640,4111757000,미확인,,,False,,\r\n경기도 용인시수지구 상현3동,31193630,4146558500,미확인,,,False,,\r\n경기도 성남시중원구 상대원2동,31022580,4113358000,미확인,,,False,,\r\n경기도 고양시덕양구 흥도동,31101530,4128153000,미확인,,,False,,\r\n경기도 수원시장안구 조원1동,31011600,4111159700,미확인,,,False,,\r\n경기도 성남시중원구 상대원3동,31022590,4113359000,미확인,,,False,,\r\n경기도 광명시 광명6동,31060560,4121057000,미확인,,,False,,\r\n경기도 의정부시 신곡2동,31030580,4115056800,미확인,,,False,,\r\n경기도 오산시 초평동,31140560,4137056000,미확인,,,False,,\r\n경기도 용인시수지구 상현2동,31193580,4146558000,미확인,,,False,,\r\n경기도 용인시기흥구 상갈동,31192650,4146353000,미확인,,,False,,\r\n경기도 성남시수정구 복정동,31021670,4113162000,미확인,,,False,,\r\n경기도 부천시소사구 송내2동,31052541,4119460000,미확인,,,False,,\r\n경기도 용인시기흥구 구갈동,31192520,4146352000,미확인,,,False,,\r\n경기도 평택시 원평동,31070600,4122060000,미확인,,,False,,\r\n경기도 안양시만안구 안양5동,31041550,4117155000,미확인,,,False,,\r\n경기도 성남시수정구 위례동,31021680,4113162500,미확인,,,False,,\r\n경기도 광명시 일직동,31060700,4121065500,미확인,,,False,,\r\n경기도 고양시덕양구 성사1동,31101540,4128154000,미확인,,,False,,\r\n경기도 수원시영통구 광교2동,31014630,4111761000,확인,DRT008,,False,광교1·2동,\r\n경기도 오산시 남촌동,31140530,4137053000,미확인,,,False,,\r\n경기도 성남시수정구 태평3동,31021560,4113156000,미확인,,,False,,\r\n경기도 고양시일산동구 중산2동,31103512,4128552600,미확인,,,False,,\r\n경기도 안산시상록구 이동,31091740,4127151500,미확인,,,False,,\r\n경기도 고양시덕양구 대덕동,31101680,4128167000,후보,,DRT014,True,덕은동,https://www.goyang.go.kr/dygu/dygu01/dygu01_5/dygu01_5_3.jsp\r\n경기도 오산시 신장1동,31140571,4137054300,미확인,,,False,,\r\n경기도 부천시원미구 원미2동,31051541,4119255000,미확인,,,False,,\r\n경기도 고양시덕양구 행신2동,31101660,4128165000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄8동,31244748,4159758000,후보,,DRT015,True,동탄신도시,\r\n경기도 고양시일산서구 탄현2동,31104512,4128754600,미확인,,,False,,\r\n경기도 성남시수정구 태평1동,31021540,4113154000,미확인,,,False,,\r\n경기도 성남시수정구 수진2동,31021590,4113158000,미확인,,,False,,\r\n경기도 의정부시 송산2동,31030700,4115057600,후보,,DRT045,True,민락1지구,https://www.ui4u.go.kr/cscportal/songsan2/main.do?mId=0311010000\r\n경기도 성남시수정구 신흥3동,31021530,4113153000,미확인,,,False,,\r\n경기도 수원시팔달구 행궁동,31013720,4111574000,미확인,,,False,,\r\n경기도 김포시 마산동,31230640,4157057500,미확인,,,False,,\r\n경기도 시흥시 정왕본동,31150640,4139058900,미확인,,,False,,\r\n경기도 부천시소사구 소사본1동,31052561,4119454000,미확인,,,False,,\r\n경기도 고양시덕양구 능곡동,31101610,4128161000,미확인,,,False,,\r\n경기도 광명시 광명2동,31060520,4121052000,미확인,,,False,,\r\n경기도 수원시팔달구 화서1동,31013700,4111568000,미확인,,,False,,\r\n경기도 용인시기흥구 영덕1동,31192630,4146351600,미확인,,,False,,\r\n경기도 수원시권선구 입북동,31012620,4111370000,확인,DRT010,,False,당수동,https://www.suwon.go.kr/sw-www/www05/www05-01/www05-01-08.jsp | https://www.suwon.go.kr/web/board/BD_board.view.do?bbsCd=1043&seq=20250617144318364\r\n경기도 김포시 풍무동,31230540,4157055000,확인,DRT019,,False,"고촌읍, 풍무동",\r\n경기도 성남시분당구 백현동,31023760,4113565700,미확인,,,False,,\r\n경기도 안산시단원구 선부2동,31092690,4127359000,미확인,,,False,,\r\n경기도 오산시 대원2동,31140582,4137052600,미확인,,,False,,\r\n경기도 고양시일산서구 덕이동,31104513,4128760000,미확인,,,False,,\r\n경기도 부천시소사구 송내1동,31052531,4119459000,미확인,,,False,,\r\n경기도 성남시중원구 금광1동,31022530,4113353000,미확인,,,False,,\r\n경기도 수원시영통구 광교1동,31014620,4111760000,확인,DRT008,,False,광교1·2동,\r\n경기도 평택시 서정동,31070520,4122052000,미확인,,,False,,\r\n경기도 광명시 광명5동,31060550,4121056000,미확인,,,False,,\r\n경기도 남양주시 다산2동,31130590,4136056500,미확인,,,False,,\r\n경기도 용인시수지구 상현1동,31193620,4146557000,미확인,,,False,,\r\n경기도 수원시팔달구 우만1동,31013540,4111571000,미확인,,,False,,\r\n경기도 안산시상록구 사이동,31091770,4127153200,미확인,,,False,,\r\n경기도 부천시소사구 범박동,31052571,4119455000,확인,DRT034,,False,"옥길동, 범박동",\r\n경기도 수원시팔달구 매산동,31013680,4111566000,미확인,,,False,,\r\n경기도 의정부시 의정부2동,31030520,4115052000,미확인,,,False,,\r\n경기도 평택시 지산동,31070550,4122055000,미확인,,,False,,\r\n경기도 고양시일산서구 송포동,31104580,4128758000,미확인,,,False,,\r\n경기도 시흥시 신천동,31150520,4139052000,미확인,,,False,,\r\n경기도 김포시 고촌읍,31230120,4157025300,확인,DRT019,,False,"고촌읍, 풍무동",\r\n경기도 군포시 송부동,31160640,4141063000,미확인,,,False,,\r\n경기도 구리시 수택2동,31120570,4131058000,미확인,,,False,,\r\n경기도 안산시단원구 백운동,31092780,4127355500,미확인,,,False,,\r\n경기도 부천시소사구 옥길동,31052581,4119456000,확인,DRT034,,False,"옥길동, 범박동",\r\n경기도 수원시장안구 송죽동,31011590,4111159100,미확인,,,False,,\r\n경기도 고양시일산동구 마두1동,31103560,4128556000,미확인,,,False,,\r\n경기도 군포시 군포2동,31160520,4141052000,미확인,,,False,,\r\n경기도 광명시 소하1동,31060660,4121064000,미확인,,,False,,\r\n경기도 용인시수지구 풍덕천1동,31193510,4146551000,미확인,,,False,,\r\n경기도 부천시소사구 역곡3동,31052601,4119458000,미확인,,,False,,\r\n경기도 군포시 산본1동,31160540,4141054000,미확인,,,False,,\r\n경기도 의왕시 내손2동,31170550,4143055000,미확인,,,False,,\r\n경기도 안산시단원구 선부1동,31092680,4127358000,미확인,,,False,,\r\n경기도 김포시 사우동,31230530,4157054000,미확인,,,False,,\r\n경기도 광명시 하안2동,31060630,4121063200,미확인,,,False,,\r\n경기도 고양시덕양구 삼송2동,31101513,4128157700,미확인,,,False,,\r\n경기도 수원시팔달구 고등동,31013690,4111567000,미확인,,,False,,\r\n경기도 시흥시 거북섬동,31150760,4139065000,미확인,,,False,,\r\n경기도 시흥시 정왕2동,31150611,4139059200,미확인,,,False,,\r\n경기도 광주시 경안동,31250540,4161051000,미확인,,,False,,\r\n경기도 부천시오정구 고강본동,31053521,4119654000,확인,DRT033,,False,"고강본동, 고강1동",\r\n경기도 구리시 수택3동,31120580,4131059000,미확인,,,False,,\r\n경기도 수원시권선구 금곡동,31012650,4111366200,미확인,,,False,,\r\n경기도 안산시상록구 본오1동,31091540,4127154000,미확인,,,False,,\r\n경기도 용인시수지구 풍덕천2동,31193520,4146552000,미확인,,,False,,\r\n경기도 고양시일산서구 일산2동,31104520,4128752000,미확인,,,False,,\r\n경기도 구리시 갈매동,31120510,4131051000,미확인,,,False,,\r\n경기도 안산시상록구 본오3동,31091560,4127156000,미확인,,,False,,\r\n경기도 수원시권선구 구운동,31012570,4111365000,미확인,,,False,,\r\n경기도 광명시 광명3동,31060530,4121054000,미확인,,,False,,\r\n경기도 부천시원미구 도당동,31051601,4119260000,미확인,,,False,,\r\n경기도 평택시 중앙동,31070510,4122051000,미확인,,,False,,\r\n경기도 광명시 광명4동,31060540,4121055000,미확인,,,False,,\r\n경기도 부천시소사구 괴안동,31052591,4119457000,미확인,,,False,,\r\n경기도 성남시분당구 수내2동,31023540,4113553000,미확인,,,False,,\r\n경기도 부천시오정구 오정동,31053561,4119656000,미확인,,,False,,\r\n경기도 하남시 신장2동,31180530,4145053000,미확인,,,False,,\r\n경기도 수원시권선구 호매실동,31012660,4111366400,미확인,,,False,,\r\n경기도 안양시동안구 범계동,31042650,4117361000,미확인,,,False,,\r\n경기도 용인시수지구 죽전2동,31193550,4146555000,미확인,,,False,,\r\n경기도 오산시 중앙동,31140510,4137051000,미확인,,,False,,\r\n경기도 하남시 덕풍3동,31180560,4145056000,미확인,,,False,,\r\n경기도 고양시덕양구 행신3동,31101514,4128165500,미확인,,,False,,\r\n경기도 수원시권선구 평동,31012550,4111355000,확인,DRT009,,False,고색동(델타플렉스),https://www.suwon.go.kr/sw-www/www05/www05-01/www05-01-08.jsp | https://www.suwon.go.kr/web/board/BD_board.view.do?bbsCd=1043&seq=20250617144318364\r\n경기도 고양시덕양구 화정1동,31101620,4128162100,미확인,,,False,,\r\n경기도 고양시일산동구 식사동,31103510,4128551000,확인,DRT012,,False,식사동,\r\n경기도 성남시분당구 야탑1동,31023620,4113562000,미확인,,,False,,\r\n경기도 평택시 통복동,31070610,4122061000,미확인,,,False,,\r\n경기도 부천시원미구 중3동,31051661,4119265000,미확인,,,False,,\r\n경기도 과천시 갈현동,31110521,4129052000,미확인,,,False,,\r\n경기도 고양시덕양구 행신4동,31101515,4128165600,미확인,,,False,,\r\n경기도 안양시동안구 갈산동,31042670,4117363000,미확인,,,False,,\r\n경기도 수원시팔달구 인계동,31013560,4111573000,미확인,,,False,,\r\n경기도 수원시영통구 원천동,31014600,4111755000,미확인,,,False,,\r\n경기도 고양시일산동구 풍산동,31103540,4128554000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄7동,31244747,4159757000,후보,,DRT015,True,동탄신도시,\r\n경기도 안양시만안구 박달동,31041660,4117164000,미확인,,,False,,\r\n경기도 성남시분당구 야탑2동,31023640,4113563000,미확인,,,False,,\r\n경기도 구리시 수택1동,31120560,4131057000,미확인,,,False,,\r\n경기도 고양시덕양구 행신1동,31101650,4128164000,미확인,,,False,,\r\n경기도 시흥시 정왕3동,31150620,4139059300,미확인,,,False,,\r\n경기도 평택시 세교동,31070640,4122064000,미확인,,,False,,\r\n경기도 파주시 운정5동,31200515,4148059000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 부천시원미구 중4동,31051671,4119266000,미확인,,,False,,\r\n경기도 시흥시 은행동,31150540,4139054000,미확인,,,False,,\r\n경기도 평택시 고덕동,31070690,4122066000,후보,,DRT007,True,고덕국제신도시,https://gnews.gg.go.kr/briefing/brief_gongbo_view.do?BS_CODE=S017&number=57039\r\n경기도 부천시원미구 상동,31051621,4119267000,미확인,,,False,,\r\n경기도 시흥시 정왕1동,31150650,4139059100,미확인,,,False,,\r\n경기도 광명시 광명1동,31060510,4121051000,미확인,,,False,,\r\n경기도 군포시 금정동,31160560,4141056000,미확인,,,False,,\r\n경기도 군포시 군포1동,31160510,4141051000,미확인,,,False,,\r\n경기도 성남시중원구 성남동,31022510,4113351000,미확인,,,False,,\r\n경기도 화성시효행구 기배동,31242735,4159351000,미확인,,,False,,\r\n경기도 고양시덕양구 화정2동,31101630,4128162200,미확인,,,False,,\r\n경기도 안양시만안구 안양4동,31041540,4117154000,미확인,,,False,,\r\n경기도 안양시만안구 안양1동,31041510,4117151000,미확인,,,False,,\r\n경기도 수원시장안구 율천동,31011550,4111156600,미확인,,,False,,\r\n경기도 하남시 미사3동,31180611,4145063000,미확인,,,False,,\r\n경기도 부천시원미구 중2동,31051651,4119264000,미확인,,,False,,\r\n경기도 고양시일산서구 주엽1동,31104550,4128755000,미확인,,,False,,\r\n경기도 부천시오정구 고강1동,31053531,4119655000,확인,DRT033,,False,"고강본동, 고강1동",\r\n경기도 시흥시 장곡동,31150710,4139063100,미확인,,,False,,\r\n경기도 수원시영통구 매탄3동,31014530,4111753000,미확인,,,False,,\r\n경기도 수원시팔달구 우만2동,31013550,4111572000,미확인,,,False,,\r\n경기도 안양시만안구 안양7동,31041570,4117157000,미확인,,,False,,\r\n경기도 파주시 운정6동,31200516,4148060000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 안양시동안구 호계3동,31042640,4117360000,미확인,,,False,,\r\n경기도 김포시 운양동,31230600,4157058000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄4동,31244744,4159754000,후보,,DRT015,True,동탄신도시,\r\n경기도 양주시 옥정2동,31260571,4163058000,후보,,DRT020,True,"옥정신도시, 삼숭동, 회천지구",\r\n경기도 안산시상록구 본오2동,31091550,4127155000,미확인,,,False,,\r\n경기도 김포시 장기본동,31230620,4157052500,미확인,,,False,,\r\n경기도 고양시일산서구 주엽2동,31104560,4128756000,미확인,,,False,,\r\n경기도 수원시권선구 세류3동,31012540,4111354000,미확인,,,False,,\r\n경기도 고양시일산서구 일산3동,31104530,4128753000,미확인,,,False,,\r\n경기도 수원시팔달구 매교동,31013670,4111565000,미확인,,,False,,\r\n경기도 부천시오정구 신흥동,31053571,4119657000,미확인,,,False,,\r\n경기도 부천시원미구 심곡3동,31051531,4119253000,미확인,,,False,,\r\n경기도 안양시동안구 인덕원동,31042691,4117355900,미확인,,,False,,\r\n경기도 수원시권선구 세류1동,31012520,4111352000,미확인,,,False,,\r\n경기도 성남시분당구 정자2동,31023550,4113556000,미확인,,,False,,\r\n경기도 수원시팔달구 화서2동,31013710,4111569000,미확인,,,False,,\r\n경기도 성남시분당구 서현2동,31023590,4113559000,미확인,,,False,,\r\n경기도 성남시분당구 이매2동,31023610,4113561000,미확인,,,False,,\r\n경기도 부천시원미구 상1동,31051681,4119268000,미확인,,,False,,\r\n경기도 부천시원미구 중1동,31051641,4119263000,미확인,,,False,,\r\n경기도 김포시 김포본동,31230610,4157051500,미확인,,,False,,\r\n경기도 과천시 부림동,31110540,4129054000,미확인,,,False,,\r\n경기도 광명시 하안3동,31060640,4121063300,미확인,,,False,,\r\n경기도 수원시영통구 매탄4동,31014540,4111754000,미확인,,,False,,\r\n경기도 안산시상록구 해양동,31091780,4127153700,미확인,,,False,,\r\n경기도 고양시일산동구 백석2동,31103610,4128555200,미확인,,,False,,\r\n경기도 광명시 철산3동,31060600,4121061000,미확인,,,False,,\r\n경기도 광명시 철산2동,31060590,4121060000,미확인,,,False,,\r\n경기도 부천시오정구 원종1동,31053541,4119652000,미확인,,,False,,\r\n경기도 오산시 대원1동,31140581,4137052300,미확인,,,False,,\r\n경기도 성남시분당구 삼평동,31023740,4113565500,미확인,,,False,,\r\n경기도 고양시일산서구 대화동,31104570,4128757000,미확인,,,False,,\r\n경기도 부천시원미구 상2동,31051691,4119269000,미확인,,,False,,\r\n경기도 고양시일산서구 탄현1동,31104511,4128754500,미확인,,,False,,\r\n경기도 수원시권선구 권선2동,31012640,4111368000,미확인,,,False,,\r\n경기도 수원시권선구 서둔동,31012560,4111356000,미확인,,,False,,\r\n경기도 남양주시 다산1동,31130580,4136054500,미확인,,,False,,\r\n경기도 수원시장안구 영화동,31011580,4111158000,미확인,,,False,,\r\n경기도 김포시 장기동,31230560,4157056000,미확인,,,False,,\r\n경기도 수원시영통구 매탄2동,31014520,4111752000,미확인,,,False,,\r\n경기도 안양시동안구 부흥동,31042540,4117354000,미확인,,,False,,\r\n경기도 수원시영통구 매탄1동,31014510,4111751000,미확인,,,False,,\r\n경기도 성남시분당구 수내1동,31023530,4113552000,미확인,,,False,,\r\n경기도 성남시분당구 정자동,31023780,4113554500,미확인,,,False,,\r\n경기도 고양시일산동구 백석1동,31103550,4128555100,미확인,,,False,,\r\n경기도 파주시 운정2동,31200570,4148056000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 평택시 비전2동,31070650,4122063000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄6동,31244746,4159756000,후보,,DRT015,True,동탄신도시,\r\n경기도 과천시 별양동,31110530,4129053000,미확인,,,False,,\r\n경기도 성남시분당구 정자1동,31023770,4113555000,미확인,,,False,,\r\n경기도 부천시원미구 심곡1동,31051511,4119251000,미확인,,,False,,\r\n경기도 수원시영통구 영통3동,31014660,4111758500,미확인,,,False,,\r\n경기도 부천시원미구 중동,31051611,4119262000,미확인,,,False,,\r\n경기도 파주시 금촌2동,31200520,4148052000,미확인,,,False,,\r\n경기도 안양시동안구 달안동,31042550,4117354600,미확인,,,False,,\r\n경기도 광명시 철산1동,31060580,4121059000,미확인,,,False,,\r\n경기도 고양시일산동구 장항2동,31103590,4128559000,미확인,,,False,,\r\n경기도 고양시일산동구 마두2동,31103570,4128557000,미확인,,,False,,\r\n경기도 수원시권선구 세류2동,31012530,4111353000,미확인,,,False,,\r\n경기도 하남시 신장1동,31180520,4145052000,미확인,,,False,,\r\n경기도 안양시동안구 평촌동,31042590,4117357000,미확인,,,False,,\r\n경기도 안양시동안구 비산2동,31042520,4117352000,미확인,,,False,,\r\n경기도 안산시단원구 원곡동,31092760,4127354500,미확인,,,False,,\r\n경기도 안양시동안구 신촌동,31042660,4117362000,미확인,,,False,,\r\n경기도 구리시 교문2동,31120550,4131054200,미확인,,,False,,\r\n경기도 고양시일산동구 장항1동,31103580,4128558000,미확인,,,False,,\r\n경기도 시흥시 정왕4동,31150720,4139059400,미확인,,,False,,\r\n경기도 수원시장안구 정자3동,31011620,4111157300,미확인,,,False,,\r\n경기도 수원시영통구 망포1동,31014670,4111759300,미확인,,,False,,\r\n경기도 광명시 하안4동,31060650,4121063400,미확인,,,False,,\r\n경기도 수원시장안구 정자1동,31011560,4111157100,미확인,,,False,,\r\n경기도 안양시동안구 귀인동,31042610,4117357800,미확인,,,False,,\r\n경기도 부천시원미구 약대동,31051631,4119261000,미확인,,,False,,\r\n경기도 부천시원미구 상3동,31051701,4119270000,미확인,,,False,,\r\n경기도 하남시 미사1동,31180620,4145061000,미확인,,,False,,\r\n경기도 하남시 미사2동,31180630,4145062000,미확인,,,False,,\r\n경기도 부천시오정구 원종2동,31053551,4119653000,미확인,,,False,,\r\n경기도 화성시병점구 병점2동,31243738,4159553000,미확인,,,False,,\r\n경기도 평택시 용이동,31070660,4122063500,미확인,,,False,,\r\n경기도 부천시원미구 심곡2동,31051521,4119252000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄2동,31244742,4159752000,후보,,DRT015,True,동탄신도시,\r\n경기도 고양시일산동구 정발산동,31103530,4128553000,미확인,,,False,,\r\n경기도 파주시 운정1동,31200512,4148055000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 화성시병점구 반월동,31243739,4159554000,미확인,,,False,,\r\n경기도 파주시 운정3동,31200513,4148057000,후보,,DRT001,True,운정신도시·교하지구,\r\n경기도 의정부시 의정부1동,31030690,4115051000,미확인,,,False,,\r\n경기도 수원시권선구 권선1동,31012600,4111367000,미확인,,,False,,\r\n경기도 안양시동안구 부림동,31042580,4117356600,미확인,,,False,,\r\n경기도 수원시장안구 정자2동,31011570,4111157200,미확인,,,False,,\r\n경기도 화성시병점구 진안동,31243736,4159551000,미확인,,,False,,\r\n경기도 안산시단원구 호수동,31092730,4127353500,미확인,,,False,,\r\n경기도 화성시동탄구 동탄3동,31244743,4159753000,후보,,DRT015,True,동탄신도시,\r\n경기도 화성시병점구 화산동,31243740,4159555000,미확인,,,False,,\r\n경기도 안양시동안구 호계1동,31042620,4117358000,미확인,,,False,,\r\n경기도 수원시영통구 영통2동,31014650,4111758000,미확인,,,False,,\r\n경기도 김포시 구래동,31230630,4157057000,미확인,,,False,,\r\n경기도 평택시 동삭동,31070680,4122065000,미확인,,,False,,\r\n경기도 안양시동안구 평안동,31042600,4117357600,미확인,,,False,,\r\n경기도 화성시만세구 새솔동,31241730,4159151000,미확인,,,False,,\r\n경기도 화성시동탄구 동탄1동,31244741,4159751000,후보,,DRT015,True,동탄신도시,\r\n경기도 수원시권선구 곡선동,31012610,4111369000,미확인,,,False,,\r\n경기도 시흥시 배곧2동,31150750,4139059700,미확인,,,False,,\r\n경기도 수원시영통구 망포2동,31014680,4111759600,미확인,,,False,,\r\n경기도 화성시병점구 병점1동,31243737,4159552000,미확인,,,False,,\r\n경기도 시흥시 배곧1동,31150740,4139059600,미확인,,,False,,\r\n'), dtype={'emd8':str, 'emdcd':str})
drt_regions = pd.read_csv(StringIO('운행구역ID,시군,원본운행구역,정리권역명,권역총차량수,연계거점,후보관리대상\nDRT001,파주시,운정신도시·교하지구,운정1·2·3지구 + 교하지구,15,,True\nDRT002,파주시,광탄면,광탄면,3,,False\nDRT003,파주시,탄현면,탄현면,3,,False\nDRT004,파주시,운정4동,운정4동,3,,False\nDRT005,안산시,대부도,대부도,6,,False\nDRT006,안산시,스마트허브,스마트허브,10,,False\nDRT007,평택시,고덕국제신도시,고덕국제신도시,15,,True\nDRT008,수원시,광교1·2동,광교1·2동,10,,False\nDRT009,수원시,고색동(델타플렉스),고색동(델타플렉스),10,,False\nDRT010,수원시,당수동,당수동,10,,False\nDRT011,고양시,고봉동,고봉동,3,,False\nDRT012,고양시,식사동,식사동,4,,False\nDRT013,고양시,향동동,향동동,4,,False\nDRT014,고양시,덕은동,덕은동,3,,True\nDRT015,화성시,동탄신도시,동탄1~9동 권역,20,,True\nDRT016,화성시,향남1·2지구,향남1·2지구,5,,False\nDRT017,화성시,봉담읍,봉담읍,5,,False\nDRT018,화성시,남양읍·마도면,남양읍·마도면,5,,False\nDRT019,김포시,"고촌읍, 풍무동","고촌읍, 풍무동",10,,False\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",옥정권 + 삼숭동 + 회천권,15,,True\nDRT021,양주시,백석읍·광적면·양주역,백석읍·광적면,4,양주역,True\nDRT022,양주시,은현·남면·덕정역,은현면·남면,3,덕정역,True\nDRT023,양주시,장흥면,장흥면,3,,False\nDRT024,하남시,감일동,감일동,5,,False\nDRT025,하남시,위례동,위례동,4,,False\nDRT026,안성시,"서부권(공도읍,양성면,원곡면)","서부권(공도읍,양성면,원곡면)",4,,False\nDRT027,안성시,"동부권(일죽면,죽산면,삼죽면)","동부권(일죽면,죽산면,삼죽면)",4,,False\nDRT028,안성시,"북부권(보개면,대덕면,고삼면,양성면)","북부권(보개면,대덕면,고삼면,양성면)",4,,False\nDRT029,안성시,"남부권(금광면,서운면,미양면,보개면)","남부권(금광면,서운면,미양면,보개면)",3,,False\nDRT030,이천시,"시내권(관고동,중리동,증포동,창전동)","시내권(관고동,중리동,증포동,창전동)",12,,False\nDRT031,이천시,장호원읍,장호원읍,7,,False\nDRT032,이천시,율면,율면,3,,False\nDRT033,부천시,"고강본동, 고강1동","고강본동, 고강1동",3,,False\nDRT034,부천시,"옥길동, 범박동","옥길동, 범박동",4,,False\nDRT035,광주시,곤지암읍,곤지암읍,4,,False\nDRT036,광주시,도척면,도척면,4,,False\nDRT037,광주시,초월읍,초월읍,4,,False\nDRT038,광주시,퇴촌면,퇴촌면,3,,False\nDRT039,여주시,세종대왕면·중앙동·오학동,세종대왕면·중앙동·오학동,8,,False\nDRT040,여주시,북내면·강천면,북내면·강천면,2,,False\nDRT041,안양시,안양9동,안양9동,2,,False\nDRT042,연천군,"연천읍, 내산리","연천읍, 내산리",2,,False\nDRT043,의왕시,초평동,초평동,2,,False\nDRT044,용인시,남사·이동읍,남사·이동읍,6,,False\nDRT045,의정부시,민락1지구,민락1지구,4,,True\nDRT046,의정부시,민락2·고산지구,민락2·고산지구,4,,True\n'))
candidate_regions = pd.read_csv(StringIO('지자체,운행구역,운행개시일,추진근거,운행구역ID,확인_읍면동,후보_읍면동,추가확인표현,행처리상태,권역총차량수,연계거점,정리권역명\n파주시 (24대),운정신도시·교하지구,2021년 12월,규제샌드박스,DRT001,,교하동 / 운정1동 / 운정2동 / 운정3동 / 운정4동 / 운정5동 / 운정6동,,후보 관리,15,,운정1·2·3지구 + 교하지구\n평택시,고덕국제신도시,2023년 4월,규제샌드박스,DRT007,,고덕동 / 고덕면,,후보 관리,15,,고덕국제신도시\n고양시 (14대),덕은동,2024년 11월,규제샌드박스,DRT014,,화전동 / 대덕동,,후보 관리,3,,덕은동\n화성시 (35대),동탄신도시,2023년 6~7월,규제샌드박스,DRT015,,동탄1동 / 동탄2동 / 동탄3동 / 동탄4동 / 동탄5동 / 동탄6동 / 동탄7동 / 동탄8동 / 동탄9동,,후보 관리,20,,동탄1~9동 권역\n양주시 (25대),"옥정신도시, 삼숭동, 회천지구",2023년 7월,규제샌드박스,DRT020,,옥정1동 / 옥정2동 / 회천1동 / 회천2동 / 회천3동 / 양주2동,,후보 관리,15,,옥정권 + 삼숭동 + 회천권\n양주시 (25대),백석읍·광적면·양주역,2025년 6월,농어촌,DRT021,,광적면 / 백석읍,,후보 관리,4,양주역,백석읍·광적면\n양주시 (25대),은현·남면·덕정역,2025년 6월,농어촌,DRT022,,남면 / 은현면,,후보 관리,3,덕정역,은현면·남면\n의정부시,민락1지구,2025년 6월,규제샌드박스,DRT045,,송산1동 / 송산2동,,후보 관리,4,,민락1지구\n의정부시,민락2·고산지구,2025년 6월,규제샌드박스,DRT046,,송산3동 / 고산동,,후보 관리,4,,민락2·고산지구\n'))
boundary_mapping = pd.read_csv(StringIO('운행구역ID,시군,원본운행구역,원본운행개시일,근거확인일,경계기준일,원본표현,읍면_key,sggnm,emdnm,emd8,emdcd,매핑상태,근거상태,후보관리사유,변환방식,운행범위,면적포함률,근거설명,근거URL,원본근거\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 교하동,파주시,교하동,31200511,4148054000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정1동,파주시,운정1동,31200512,4148055000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정2동,파주시,운정2동,31200570,4148056000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정3동,파주시,운정3동,31200513,4148057000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정4동,파주시,운정4동,31200514,4148058000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정5동,파주시,운정5동,31200515,4148059000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT001,파주시,운정신도시·교하지구,2021년 12월,2026-10-05,2026-07-01,운정신도시·교하지구,경기도 파주시 운정6동,파주시,운정6동,31200516,4148060000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 관련 동을 검토 후보로만 제시. 지구 경계와 각 동의 실제 운행 여부 미확정.,,data/raw/경기도_똑버스_운행현황.csv\nDRT002,파주시,광탄면,2023년 12월,2026-10-05,2026-07-01,광탄면,경기도 파주시 광탄면,파주시,광탄면,31200350,4148035000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT003,파주시,탄현면,2023년 12월,2026-10-05,2026-07-01,탄현면,경기도 파주시 탄현면,파주시,탄현면,31200320,4148032000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT004,파주시,운정4동,2023년 12월,2026-10-05,2026-07-01,운정4동,경기도 파주시 운정4동,파주시,운정4동,31200514,4148058000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT005,안산시,대부도,2023년 3월,2026-10-05,2026-07-01,대부도,경기도 안산시단원구 대부동,안산시단원구,대부동,31092720,4127361000,확인,확인,,섬→행정동,하위구역 연결; 실제 서비스 경계 미확인,,대부도를 대부동에 연결하되 풍도 등 대부동의 다른 섬까지 운행한다고 확대하지 않음.,https://www.ansan.go.kr/stat/common/cntnts/selectContents.do?cntnts_id=C0001013 | https://www.ansan.go.kr/danwongu/common/cntnts/selectContents.do?cntnts_id=C0000880,data/raw/경기도_똑버스_운행현황.csv\nDRT006,안산시,스마트허브,2024년 7월,2026-10-05,2026-07-01,스마트허브,경기도 안산시단원구 초지동,안산시단원구,초지동,31092660,4127357000,확인,확인,,산업단지→법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,계획 자료의 운행구역 성곡·목내·원시·초지동과 초지동 관할 목록을 대조. 산업단지 외 연계 정류장 범위는 미검증.,https://smartcity.go.kr/wp-content/uploads/2026/02/안산시-스마트도시계획_26.2.pdf | https://www.ansan.go.kr/danwongu/common/cntnts/selectContents.do?cntnts_id=C0000914,data/raw/경기도_똑버스_운행현황.csv\nDRT007,평택시,고덕국제신도시,2023년 4월,2026-10-05,2026-07-01,고덕국제신도시,경기도 평택시 고덕동,평택시,고덕동,31070690,4122066000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,공식 운행 공지는 신도시 전역이나 행정동 경계와의 교차 범위는 없음. 관련 동·면 후보만 제시.,https://gnews.gg.go.kr/briefing/brief_gongbo_view.do?BS_CODE=S017&number=57039,data/raw/경기도_똑버스_운행현황.csv\nDRT007,평택시,고덕국제신도시,2023년 4월,2026-10-05,2026-07-01,고덕국제신도시,경기도 평택시 고덕면,평택시,고덕면,31070330,4122033000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,공식 운행 공지는 신도시 전역이나 행정동 경계와의 교차 범위는 없음. 관련 동·면 후보만 제시.,https://gnews.gg.go.kr/briefing/brief_gongbo_view.do?BS_CODE=S017&number=57039,data/raw/경기도_똑버스_운행현황.csv\nDRT008,수원시,광교1·2동,2023년 5월,2026-10-05,2026-07-01,광교1동,경기도 수원시영통구 광교1동,수원시영통구,광교1동,31014620,4111760000,확인,확인,,축약명확장,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT008,수원시,광교1·2동,2023년 5월,2026-10-05,2026-07-01,광교2동,경기도 수원시영통구 광교2동,수원시영통구,광교2동,31014630,4111761000,확인,확인,,축약명확장,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT009,수원시,고색동(델타플렉스),2024년 10월,2026-10-05,2026-07-01,고색동(델타플렉스),경기도 수원시권선구 평동,수원시권선구,평동,31012550,4111355000,확인,확인,,법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,고색동 소재 산업단지를 평동에 연결. 서비스 경계 면적은 미확인.,https://www.suwon.go.kr/sw-www/www05/www05-01/www05-01-08.jsp | https://www.suwon.go.kr/web/board/BD_board.view.do?bbsCd=1043&seq=20250617144318364,data/raw/경기도_똑버스_운행현황.csv\nDRT010,수원시,당수동,2025년 6월,2026-10-05,2026-07-01,당수동,경기도 수원시권선구 입북동,수원시권선구,입북동,31012620,4111370000,확인,확인,,법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,당수동을 관할 행정동 입북동에 연결.,https://www.suwon.go.kr/sw-www/www05/www05-01/www05-01-08.jsp | https://www.suwon.go.kr/web/board/BD_board.view.do?bbsCd=1043&seq=20250617144318364,data/raw/경기도_똑버스_운행현황.csv\nDRT011,고양시,고봉동,2023년 6월,2026-10-05,2026-07-01,고봉동,경기도 고양시일산동구 고봉동,고양시일산동구,고봉동,31103600,4128560000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT012,고양시,식사동,2023년 6월,2026-10-05,2026-07-01,식사동,경기도 고양시일산동구 식사동,고양시일산동구,식사동,31103510,4128551000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT013,고양시,향동동,2024년 9월,2026-10-05,2026-07-01,향동동,경기도 고양시덕양구 화전동,고양시덕양구,화전동,31101670,4128166000,확인,확인,,법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,관할구역 표에서 향동동은 화전동에 속함.,https://www.goyang.go.kr/dygu/dygu01/dygu01_5/dygu01_5_3.jsp,data/raw/경기도_똑버스_운행현황.csv\nDRT014,고양시,덕은동,2024년 11월,2026-10-05,2026-07-01,덕은동,경기도 고양시덕양구 화전동,고양시덕양구,화전동,31101670,4128166000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,법정동 분할관할,하위구역 연결; 실제 서비스 경계 미확인,,덕은동 일부가 두 행정동에 걸침. 실제 덕은 운행구역 도형/정류장이 없어 어느 부분을 운행하는지 확정하지 않음.,https://www.goyang.go.kr/dygu/dygu01/dygu01_5/dygu01_5_3.jsp,data/raw/경기도_똑버스_운행현황.csv\nDRT014,고양시,덕은동,2024년 11월,2026-10-05,2026-07-01,덕은동,경기도 고양시덕양구 대덕동,고양시덕양구,대덕동,31101680,4128167000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,법정동 분할관할,하위구역 연결; 실제 서비스 경계 미확인,,덕은동 일부가 두 행정동에 걸침. 실제 덕은 운행구역 도형/정류장이 없어 어느 부분을 운행하는지 확정하지 않음.,https://www.goyang.go.kr/dygu/dygu01/dygu01_5/dygu01_5_3.jsp,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄1동,화성시동탄구,동탄1동,31244741,4159751000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄2동,화성시동탄구,동탄2동,31244742,4159752000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄3동,화성시동탄구,동탄3동,31244743,4159753000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄4동,화성시동탄구,동탄4동,31244744,4159754000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄5동,화성시동탄구,동탄5동,31244745,4159755000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄6동,화성시동탄구,동탄6동,31244746,4159756000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄7동,화성시동탄구,동탄7동,31244747,4159757000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄8동,화성시동탄구,동탄8동,31244748,4159758000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT015,화성시,동탄신도시,2023년 6~7월,2026-10-05,2026-07-01,동탄신도시,경기도 화성시동탄구 동탄9동,화성시동탄구,동탄9동,31244749,4159759000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 동탄 동을 검토 후보로 제시. 동별 운행을 확정한 것이 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT016,화성시,향남1·2지구,2023년 6월,2026-10-05,2026-07-01,향남1·2지구,경기도 화성시만세구 향남읍,화성시만세구,향남읍,31241722,4159125300,확인,확인,,지구→읍면,하위구역 연결; 실제 서비스 경계 미확인,,향남1·2신도시 소재 읍을 연결.,https://www.hscity.go.kr/www/user/bbs/BD_selectBbs.do?q_bbsCode=1010&q_bbscttSn=10100001315536,data/raw/경기도_똑버스_운행현황.csv\nDRT017,화성시,봉담읍,2024년 9월,2026-10-05,2026-07-01,봉담읍,경기도 화성시효행구 봉담읍,화성시효행구,봉담읍,31242731,4159325000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT018,화성시,남양읍·마도면,2024년 9월,2026-10-05,2026-07-01,남양읍,경기도 화성시만세구 남양읍,화성시만세구,남양읍,31241723,4159125600,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT018,화성시,남양읍·마도면,2024년 9월,2026-10-05,2026-07-01,마도면,경기도 화성시만세구 마도면,화성시만세구,마도면,31241724,4159131000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT019,김포시,"고촌읍, 풍무동",2023년 6월,2026-10-05,2026-07-01,고촌읍,경기도 김포시 고촌읍,김포시,고촌읍,31230120,4157025300,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT019,김포시,"고촌읍, 풍무동",2023년 6월,2026-10-05,2026-07-01,풍무동,경기도 김포시 풍무동,김포시,풍무동,31230540,4157055000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,옥정신도시,경기도 양주시 옥정1동,양주시,옥정1동,31260561,4163057000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,옥정 동 후보. 지구 경계와 실제 운행구역 확인 필요.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,옥정신도시,경기도 양주시 옥정2동,양주시,옥정2동,31260571,4163058000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,옥정 동 후보. 지구 경계와 실제 운행구역 확인 필요.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,회천지구,경기도 양주시 회천1동,양주시,회천1동,31260530,4163053000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 회천 동 후보. 모든 회천 동이 해당 지구에 속한다는 의미가 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,회천지구,경기도 양주시 회천2동,양주시,회천2동,31260540,4163054000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 회천 동 후보. 모든 회천 동이 해당 지구에 속한다는 의미가 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,회천지구,경기도 양주시 회천3동,양주시,회천3동,31260550,4163055000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,기준 명단의 회천 동 후보. 모든 회천 동이 해당 지구에 속한다는 의미가 아님.,,data/raw/경기도_똑버스_운행현황.csv\nDRT020,양주시,"옥정신도시, 삼숭동, 회천지구",2023년 7월,2026-10-05,2026-07-01,삼숭동,경기도 양주시 양주2동,양주시,양주2동,31260520,4163052000,후보,확인,사용자 지정 9개 권역 잠정 후보 관리,법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,삼숭동을 관할 행정동 양주2동에 연결.,https://www.yangju.go.kr/www/contents.do?key=415,data/raw/경기도_똑버스_운행현황.csv\nDRT021,양주시,백석읍·광적면·양주역,2025년 6월,2026-10-05,2026-07-01,광적면,경기도 양주시 광적면,양주시,광적면,31260330,4163033000,후보,확인,사용자 지정 9개 권역 잠정 후보 관리,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT021,양주시,백석읍·광적면·양주역,2025년 6월,2026-10-05,2026-07-01,백석읍,경기도 양주시 백석읍,양주시,백석읍,31260110,4163025000,후보,확인,사용자 지정 9개 권역 잠정 후보 관리,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT022,양주시,은현·남면·덕정역,2025년 6월,2026-10-05,2026-07-01,남면,경기도 양주시 남면,양주시,남면,31260320,4163032000,후보,확인,사용자 지정 9개 권역 잠정 후보 관리,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT022,양주시,은현·남면·덕정역,2025년 6월,2026-10-05,2026-07-01,은현면,경기도 양주시 은현면,양주시,은현면,31260310,4163031000,후보,확인,사용자 지정 9개 권역 잠정 후보 관리,축약명확장,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT023,양주시,장흥면,2025년 6월,2026-10-05,2026-07-01,장흥면,경기도 양주시 장흥면,양주시,장흥면,31260340,4163034000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT024,하남시,감일동,2023년 8월,2026-10-05,2026-07-01,감일동,경기도 하남시 감일동,하남시,감일동,31180670,4145058200,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT025,하남시,위례동,2023년 8월,2026-10-05,2026-07-01,위례동,경기도 하남시 위례동,하남시,위례동,31180650,4145058500,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT026,안성시,"서부권(공도읍,양성면,원곡면)",2023년 10월,2026-10-05,2026-07-01,공도읍,경기도 안성시 공도읍,안성시,공도읍,31220110,4155025000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT026,안성시,"서부권(공도읍,양성면,원곡면)",2023년 10월,2026-10-05,2026-07-01,양성면,경기도 안성시 양성면,안성시,양성면,31220360,4155036000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT026,안성시,"서부권(공도읍,양성면,원곡면)",2023년 10월,2026-10-05,2026-07-01,원곡면,경기도 안성시 원곡면,안성시,원곡면,31220380,4155038000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT027,안성시,"동부권(일죽면,죽산면,삼죽면)",2023년 10월,2026-10-05,2026-07-01,삼죽면,경기도 안성시 삼죽면,안성시,삼죽면,31220410,4155041000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT027,안성시,"동부권(일죽면,죽산면,삼죽면)",2023년 10월,2026-10-05,2026-07-01,일죽면,경기도 안성시 일죽면,안성시,일죽면,31220390,4155039000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT027,안성시,"동부권(일죽면,죽산면,삼죽면)",2023년 10월,2026-10-05,2026-07-01,죽산면,경기도 안성시 죽산면,안성시,죽산면,31220400,4155040000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT028,안성시,"북부권(보개면,대덕면,고삼면,양성면)",2024년 5월,2026-10-05,2026-07-01,고삼면,경기도 안성시 고삼면,안성시,고삼면,31220420,4155042000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT028,안성시,"북부권(보개면,대덕면,고삼면,양성면)",2024년 5월,2026-10-05,2026-07-01,대덕면,경기도 안성시 대덕면,안성시,대덕면,31220350,4155035000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT028,안성시,"북부권(보개면,대덕면,고삼면,양성면)",2024년 5월,2026-10-05,2026-07-01,보개면,경기도 안성시 보개면,안성시,보개면,31220310,4155031000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT028,안성시,"북부권(보개면,대덕면,고삼면,양성면)",2024년 5월,2026-10-05,2026-07-01,양성면,경기도 안성시 양성면,안성시,양성면,31220360,4155036000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT029,안성시,"남부권(금광면,서운면,미양면,보개면)",2024년 5월,2026-10-05,2026-07-01,금광면,경기도 안성시 금광면,안성시,금광면,31220320,4155032000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT029,안성시,"남부권(금광면,서운면,미양면,보개면)",2024년 5월,2026-10-05,2026-07-01,미양면,경기도 안성시 미양면,안성시,미양면,31220340,4155034000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT029,안성시,"남부권(금광면,서운면,미양면,보개면)",2024년 5월,2026-10-05,2026-07-01,보개면,경기도 안성시 보개면,안성시,보개면,31220310,4155031000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT029,안성시,"남부권(금광면,서운면,미양면,보개면)",2024년 5월,2026-10-05,2026-07-01,서운면,경기도 안성시 서운면,안성시,서운면,31220330,4155033000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT030,이천시,"시내권(관고동,중리동,증포동,창전동)",2023년 12월,2026-10-05,2026-07-01,관고동,경기도 이천시 관고동,이천시,관고동,31210530,4150053000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT030,이천시,"시내권(관고동,중리동,증포동,창전동)",2023년 12월,2026-10-05,2026-07-01,중리동,경기도 이천시 중리동,이천시,중리동,31210520,4150052000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT030,이천시,"시내권(관고동,중리동,증포동,창전동)",2023년 12월,2026-10-05,2026-07-01,증포동,경기도 이천시 증포동,이천시,증포동,31210540,4150051500,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT030,이천시,"시내권(관고동,중리동,증포동,창전동)",2023년 12월,2026-10-05,2026-07-01,창전동,경기도 이천시 창전동,이천시,창전동,31210510,4150051000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT031,이천시,장호원읍,2023년 12월,2026-10-05,2026-07-01,장호원읍,경기도 이천시 장호원읍,이천시,장호원읍,31210110,4150025000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT032,이천시,율면,2023년 12월,2026-10-05,2026-07-01,율면,경기도 이천시 율면,이천시,율면,31210380,4150038000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT033,부천시,"고강본동, 고강1동",2024년 4월,2026-10-05,2026-07-01,고강1동,경기도 부천시오정구 고강1동,부천시오정구,고강1동,31053531,4119655000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT033,부천시,"고강본동, 고강1동",2024년 4월,2026-10-05,2026-07-01,고강본동,경기도 부천시오정구 고강본동,부천시오정구,고강본동,31053521,4119654000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT034,부천시,"옥길동, 범박동",2024년 4월,2026-10-05,2026-07-01,범박동,경기도 부천시소사구 범박동,부천시소사구,범박동,31052571,4119455000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT034,부천시,"옥길동, 범박동",2024년 4월,2026-10-05,2026-07-01,옥길동,경기도 부천시소사구 옥길동,부천시소사구,옥길동,31052581,4119456000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT035,광주시,곤지암읍,2024년 5월,2026-10-05,2026-07-01,곤지암읍,경기도 광주시 곤지암읍,광주시,곤지암읍,31250140,4161025900,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT036,광주시,도척면,2024년 5월,2026-10-05,2026-07-01,도척면,경기도 광주시 도척면,광주시,도척면,31250340,4161033000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT037,광주시,초월읍,2024년 6월,2026-10-05,2026-07-01,초월읍,경기도 광주시 초월읍,광주시,초월읍,31250120,4161025300,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT038,광주시,퇴촌면,2024년 6월,2026-10-05,2026-07-01,퇴촌면,경기도 광주시 퇴촌면,광주시,퇴촌면,31250350,4161034000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT039,여주시,세종대왕면·중앙동·오학동,2024년 6월,2026-10-05,2026-07-01,세종대왕면,경기도 여주시 세종대왕면,여주시,세종대왕면,31280390,4167034500,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT039,여주시,세종대왕면·중앙동·오학동,2024년 6월,2026-10-05,2026-07-01,오학동,경기도 여주시 오학동,여주시,오학동,31280530,4167053000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT039,여주시,세종대왕면·중앙동·오학동,2024년 6월,2026-10-05,2026-07-01,중앙동,경기도 여주시 중앙동,여주시,중앙동,31280520,4167052000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT040,여주시,북내면·강천면,2024년 6월,2026-10-05,2026-07-01,강천면,경기도 여주시 강천면,여주시,강천면,31280380,4167037000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT040,여주시,북내면·강천면,2024년 6월,2026-10-05,2026-07-01,북내면,경기도 여주시 북내면,여주시,북내면,31280370,4167036000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT041,안양시,안양9동,2024년 8월,2026-10-05,2026-07-01,안양9동,경기도 안양시만안구 병목안동,안양시만안구,병목안동,31041672,4117158300,확인,확인,,명칭변경,전역여부 미확인,,2026-07-01 행정동 명칭 변경을 기준 경계에 반영.,https://www.anyang.go.kr/main/contents.do?key=289,data/raw/경기도_똑버스_운행현황.csv\nDRT042,연천군,"연천읍, 내산리",2024년 8월,2026-10-05,2026-07-01,내산리,경기도 연천군 신서면,연천군,신서면,31550360,4180036000,확인,확인,,리→읍면,하위구역 연결; 실제 서비스 경계 미확인,,내산리의 소속 면을 연결. 신서면 전역 운행 의미가 아님.,https://yeoncheon.go.kr/www/contents.do?key=3410 | https://www.yeoncheon.go.kr/www/contents.do?key=3441,data/raw/경기도_똑버스_운행현황.csv\nDRT042,연천군,"연천읍, 내산리",2024년 8월,2026-10-05,2026-07-01,연천읍,경기도 연천군 연천읍,연천군,연천읍,31550110,4180025000,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT043,의왕시,초평동,2025년 6월,2026-10-05,2026-07-01,초평동,경기도 의왕시 부곡동,의왕시,부곡동,31170520,4143052000,확인,확인,,법정동→행정동,하위구역 연결; 실제 서비스 경계 미확인,,초평동 전체의 관할 행정동은 부곡동. 부곡동 전체가 운행구역이라는 뜻은 아님.,https://www.uiwang.go.kr/UWKORINTRO0104,data/raw/경기도_똑버스_운행현황.csv\nDRT044,용인시,남사·이동읍,2025년 6월,2026-10-05,2026-07-01,남사읍,경기도 용인시처인구 남사읍,용인시처인구,남사읍,31191140,4146125900,확인,확인,,축약명확장,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT044,용인시,남사·이동읍,2025년 6월,2026-10-05,2026-07-01,이동읍,경기도 용인시처인구 이동읍,용인시처인구,이동읍,31191130,4146125600,확인,확인,,명칭일치,전역여부 미확인,,CSV에 읍면동 명시. 행정동 전체를 운행한다는 판단은 하지 않음.,,data/raw/경기도_똑버스_운행현황.csv\nDRT045,의정부시,민락1지구,2025년 6월,2026-10-05,2026-07-01,민락1지구,경기도 의정부시 송산1동,의정부시,송산1동,31030591,4115057300,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,법정동과 지구 경계 분할 확인이 필요하여 검토 후보로만 유지.,https://www.ui4u.go.kr/cscportal/songsan2/main.do?mId=0311010000,data/raw/경기도_똑버스_운행현황.csv\nDRT045,의정부시,민락1지구,2025년 6월,2026-10-05,2026-07-01,민락1지구,경기도 의정부시 송산2동,의정부시,송산2동,31030700,4115057600,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,법정동과 지구 경계 분할 확인이 필요하여 검토 후보로만 유지.,https://www.ui4u.go.kr/cscportal/songsan2/main.do?mId=0311010000,data/raw/경기도_똑버스_운행현황.csv\nDRT046,의정부시,민락2·고산지구,2025년 6월,2026-10-05,2026-07-01,민락2·고산지구,경기도 의정부시 송산3동,의정부시,송산3동,31030710,4115057800,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,민락2지구와 송산3동의 관계는 공식 자료에 나타나나 합쳐진 운행구역의 경계와 고산지구 범위는 미검증. 후보로 유지.,https://www.ui4u.go.kr/cscportal/bbs/view.do?bIdx=348257&mId=0402000000&ptIdx=85,data/raw/경기도_똑버스_운행현황.csv\nDRT046,의정부시,민락2·고산지구,2025년 6월,2026-10-05,2026-07-01,민락2·고산지구,경기도 의정부시 고산동,의정부시,고산동,31030721,4115063000,후보,후보,사용자 지정 9개 권역 잠정 후보 관리,생활권 후보,하위구역 연결; 실제 서비스 경계 미확인,,민락2지구와 송산3동의 관계는 공식 자료에 나타나나 합쳐진 운행구역의 경계와 고산지구 범위는 미검증. 후보로 유지.,https://www.ui4u.go.kr/cscportal/bbs/view.do?bIdx=348257&mId=0402000000&ptIdx=85,data/raw/경기도_똑버스_운행현황.csv\n'), dtype={'emd8':str, 'emdcd':str})
assert set(gyeonggi_drt['읍면_key']) == set(boundary_links['읍면_key']), '경계 목록이 변경됨: 대응표 재생성 필요'
boundary_result = gyeonggi_drt.merge(boundary_links, on='읍면_key', validate='one_to_one')
display(candidate_regions)
display(boundary_result.head(20)[['경기도순위_잠정','sggnm','emdnm','이동격차지수','경계기준_DRT연결','후보연결존재']])
boundary_summary = pd.DataFrame([{
    '상위N':n,
    '확인':int(boundary_result.head(n)['경계기준_DRT연결'].eq('확인').sum()),
    '후보만존재':int(boundary_result.head(n)['경계기준_DRT연결'].eq('후보').sum()),
    '후보연결존재':int(boundary_result.head(n)['후보연결존재'].sum())
} for n in (10,20,50,len(boundary_result))])
display(boundary_summary)
from pathlib import Path
candidate_out = Path('drt_boundary_mapping')
candidate_out.mkdir(exist_ok=True)
for name, frame in [('경기도_경계기준_DRT비교',boundary_result),('후보9개_권역검토표',candidate_regions),('똑버스_운행권역표',drt_regions),('똑버스_읍면동_대응표',boundary_mapping),('경계변환_요약',boundary_summary)]:
    frame.to_csv(candidate_out / (name+'.csv'), index=False, encoding='utf-8-sig')


In [ ]:
shutil.make_archive('drt_boundary_mapping', 'zip', 'drt_boundary_mapping')
files.download('drt_boundary_mapping.zip')
